
# Experimento COBYLA em subespaço ativo — fiel ao `01_generate_dataset`

## Objetivo

Este notebook mantém **o mesmo problema, os mesmos dados, o mesmo Hamiltoniano de Ising e o mesmo ansatz parametrizado de Dicke** do código original.

A única mudança experimental é controlar quais componentes do vetor

\[
\boldsymbol{\theta}
=
(\theta_0,\theta_1,\ldots,\theta_{29})
\]

o COBYLA pode modificar.

O experimento compara:

- COBYLA completo: 30 parâmetros ativos;
- somente \(\theta_{17}\);
- \(\theta_{17}\) e \(\theta_{14}\);
- Top-4: \(\theta_2,\theta_3,\theta_{14},\theta_{17}\);
- quatro parâmetros de baixo contraste;
- quatro parâmetros aleatórios;
- Top-4 seguido de refinamento com os 30 parâmetros;
- curva de dimensão ativa: 1, 2, 4, 6, 8, 12, 16 e 30.

## Correções em relação às versões anteriores

1. `DickeStateAnsatz` é uma **função**, não uma classe do `qiskit-finance`, e agora está definida diretamente neste notebook.
2. O retorno usado no código original é:

```python
assets_returns = daily_returns.sum()
```

e não a média.
3. A função objetivo original é:

\[
q\,x^\top\Sigma x
-
(1-q)\,x^\top\mu
+
r_f.
\]

4. O `assign_parameters` é utilizado exatamente no ponto correto: **depois da otimização**, para transformar o circuito simbólico em um circuito numérico que pode ser medido.
5. Na otimização reduzida, o resultado do COBYLA contém apenas os parâmetros ativos. Antes de chamar `assign_parameters`, o notebook reconstrói obrigatoriamente o vetor completo com 30 valores.
6. O notebook é **autônomo**: não abre, não lê e não extrai código de outro `.ipynb`.



## Dependências

Execute esta instalação apenas quando os pacotes não estiverem disponíveis:

```python
# %pip install --upgrade \
#     numpy pandas matplotlib cvxpy pyscipopt \
#     docplex qiskit qiskit-aer \
#     qiskit-optimization qiskit-algorithms
```

Depois da instalação, reinicie o kernel.


In [ ]:

# ============================================================
# 1. IMPORTS E CONFIGURAÇÃO
# ============================================================

from __future__ import annotations

from itertools import combinations
from pathlib import Path
from time import perf_counter
from typing import Any, Iterable
import json
import math
import warnings

import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from docplex.mp.model import Model

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.primitives import BackendEstimatorV2
from qiskit_aer import AerSimulator

from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.converters import QuadraticProgramToQubo

try:
    from qiskit_algorithms.minimum_eigensolvers import (
        NumPyMinimumEigensolver,
    )
    from qiskit_algorithms.optimizers import COBYLA
except ImportError:
    from qiskit.algorithms.minimum_eigensolvers import (
        NumPyMinimumEigensolver,
    )
    from qiskit.algorithms.optimizers import COBYLA


# ------------------------------------------------------------
# Problema fixo
# ------------------------------------------------------------

stock_path = Path("../data/assets/stocks_prices.csv")

n_assets = 10
budget = 4

q_value = 0.5
risk_free = 0.0475

known_exact_bitstring = "1001011000"
known_exact_energy = -0.8181062577496281

# ------------------------------------------------------------
# Configuração do experimento
# ------------------------------------------------------------

fast_mode = True

n_restarts = 3 if fast_mode else 20
maxiter_main = 250 if fast_mode else 1000
maxiter_refinement = 80 if fast_mode else 300

shots = 4096
random_seed = 42
near_optimal_threshold = 1e-3

run_dimension_curve = True
run_two_stage = True

output_dir = Path(
    "results/cobyla_subespaco_ativo_fiel_original"
)
output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

rng = np.random.default_rng(
    random_seed
)

print("arquivo de preços:", stock_path)
print("problema:", f"{n_assets} ativos, escolher {budget}")
print("q:", q_value)
print("taxa livre de risco:", risk_free)
print("bitstring conhecido:", known_exact_bitstring)
print("modo rápido:", fast_mode)



# Parte I — reconstrução idêntica do problema original

O problema financeiro é reconstruído uma única vez. Durante todo o experimento:

- os preços não mudam;
- os retornos não mudam;
- a covariância não muda;
- o QUBO não muda;
- o Hamiltoniano de Ising não muda;
- o ansatz não muda.

Somente o conjunto de parâmetros liberados para o COBYLA é alterado.


In [ ]:

# ============================================================
# 2. PREÇOS -> RETORNOS DIÁRIOS -> RETORNO DOS ATIVOS
# ============================================================

if not stock_path.exists():
    raise FileNotFoundError(
        "O arquivo não foi encontrado em:\n"
        f"{stock_path.resolve()}"
    )

stocks_prices = pd.read_csv(
    "../data/assets/stocks_prices.csv",
    index_col=0,
)

# Garante que somente dados numéricos permaneçam.
stocks_prices = (
    stocks_prices
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .dropna(
        axis=1,
        how="all",
    )
    .ffill()
    .bfill()
    .dropna()
)

if stocks_prices.shape[1] != n_assets:
    raise ValueError(
        f"Esperados {n_assets} ativos, mas o CSV possui "
        f"{stocks_prices.shape[1]} colunas numéricas."
    )

# CÓDIGO ORIGINAL:
# daily_returns = stocks_prices.pct_change().dropna()
daily_returns = (
    stocks_prices
    .pct_change(
        fill_method=None
    )
    .dropna()
)

# PONTO FUNDAMENTAL:
# O notebook original usa a SOMA dos retornos diários.
# Não trocar por mean() neste experimento, pois isso mudaria
# o problema, o Ising, a energia e possivelmente o bitstring.
assets_returns = daily_returns.sum()

# Matriz de covariância exatamente como no original.
covariance = daily_returns.cov()

# Simetrização numérica sem alterar a formulação.
covariance_values = covariance.to_numpy(
    dtype=float
)
covariance_values = 0.5 * (
    covariance_values
    + covariance_values.T
)

assets_return_values = assets_returns.to_numpy(
    dtype=float
)

tickers = list(
    stocks_prices.columns
)

print("tickers:", tickers)
print("preços:", stocks_prices.shape)
print("retornos diários:", daily_returns.shape)
print("retornos acumulados:", assets_returns.shape)
print("covariância:", covariance.shape)

display(
    assets_returns.to_frame(
        "assets_return"
    )
)

display(covariance)


In [ ]:

# ============================================================
# 3. PROBLEMA CLÁSSICO COM CVXPY
# ============================================================

# Parâmetro original de ponderação risco-retorno.
q_cvx = cp.Parameter(
    nonneg=True,
    name="q",
)
q_cvx.value = q_value

# Uma variável booleana para cada ativo:
# 0 = não selecionado
# 1 = selecionado
var_x = cp.Variable(
    shape=(covariance.shape[0],),
    boolean=True,
    name="portfolio_selection",
)

# OBJETIVO ORIGINAL:
#
# q * risco
# - (1-q) * retorno
# + taxa livre de risco
#
# O termo risk_free é constante. Ele desloca a energia,
# mas não altera qual carteira minimiza a função.
objective_cvx = cp.Minimize(
    q_cvx
    * cp.quad_form(
        var_x,
        cp.psd_wrap(
            covariance_values
        ),
    )
    - (1 - q_cvx)
    * var_x
    @ assets_return_values
    + risk_free
)

# Restrição: selecionar exatamente quatro ativos.
constraints_cvx = [
    np.ones(
        shape=(n_assets,)
    )
    @ var_x
    == budget
]

problem_cvx = cp.Problem(
    objective=objective_cvx,
    constraints=constraints_cvx,
)

cvxpy_status = None
cvxpy_solution = None
cvxpy_value = None

installed_solvers = set(
    cp.installed_solvers()
)

if "SCIP" in installed_solvers:
    cvxpy_value = problem_cvx.solve(
        solver="SCIP"
    )
    cvxpy_status = problem_cvx.status

    if var_x.value is not None:
        cvxpy_solution = np.rint(
            np.asarray(
                var_x.value
            ).reshape(-1)
        ).astype(int)
else:
    warnings.warn(
        "O SCIP não está disponível no CVXPY. "
        "A enumeração exata das 210 carteiras será usada "
        "como validação clássica."
    )

print("status CVXPY:", cvxpy_status)
print("valor CVXPY:", cvxpy_value)
print("solução CVXPY:", cvxpy_solution)


In [ ]:

# ============================================================
# 4. ENUMERAÇÃO EXATA DAS C(10,4)=210 CARTEIRAS
# ============================================================

def original_classical_objective(
    x_binary,
):
    """
    Mesma função objetivo utilizada em CVXPY e DOcplex.
    """
    x_binary = np.asarray(
        x_binary,
        dtype=float,
    ).reshape(-1)

    risk_term = (
        q_value
        * x_binary
        @ covariance_values
        @ x_binary
    )

    return_term = (
        (1 - q_value)
        * x_binary
        @ assets_return_values
    )

    return float(
        risk_term
        - return_term
        + risk_free
    )


enumeration_rows = []

for selected_indices in combinations(
    range(n_assets),
    budget,
):
    x_candidate = np.zeros(
        n_assets,
        dtype=int,
    )
    x_candidate[
        list(selected_indices)
    ] = 1

    bitstring_asset_order = "".join(
        str(int(value))
        for value in x_candidate
    )

    # O Qiskit exibe q_(n-1)...q_0.
    bitstring_qiskit_order = (
        bitstring_asset_order[::-1]
    )

    enumeration_rows.append(
        {
            "selected_indices": selected_indices,
            "x_asset_order": x_candidate,
            "bitstring_asset_order": bitstring_asset_order,
            "bitstring_qiskit_order": bitstring_qiskit_order,
            "objective": original_classical_objective(
                x_candidate
            ),
        }
    )

enumeration_df = (
    pd.DataFrame(
        enumeration_rows
    )
    .sort_values(
        "objective"
    )
    .reset_index(
        drop=True
    )
)

enumeration_best = enumeration_df.iloc[0]

classical_exact_value = float(
    enumeration_best[
        "objective"
    ]
)

classical_exact_x = np.asarray(
    enumeration_best[
        "x_asset_order"
    ],
    dtype=int,
)

classical_asset_bitstring = str(
    enumeration_best[
        "bitstring_asset_order"
    ]
)

classical_qiskit_bitstring = str(
    enumeration_best[
        "bitstring_qiskit_order"
    ]
)

selected_tickers = [
    ticker
    for ticker, selected
    in zip(
        tickers,
        classical_exact_x,
    )
    if selected == 1
]

print("carteiras avaliadas:", len(enumeration_df))
print("valor clássico exato:", classical_exact_value)
print("vetor em ordem dos ativos:", classical_exact_x)
print("bitstring em ordem dos ativos:", classical_asset_bitstring)
print("bitstring em ordem Qiskit:", classical_qiskit_bitstring)
print("ativos selecionados:", selected_tickers)

display(
    enumeration_df[
        [
            "selected_indices",
            "bitstring_asset_order",
            "bitstring_qiskit_order",
            "objective",
        ]
    ].head(10)
)


In [ ]:

# ============================================================
# 5. DOCPLEX -> QUADRATIC PROGRAM -> QUBO -> ISING
# ============================================================

model = Model(
    name="portfolio_optimization"
)

variables = np.array(
    [
        model.binary_var(
            name=f"x_{index}"
        )
        for index in range(n_assets)
    ],
    dtype=object,
)

# No código original a expressão foi escrita de forma compacta:
#
# variables @ covariance.values @ variables
#
# A soma explícita abaixo é matematicamente idêntica, mas evita
# problemas entre versões do NumPy e objetos simbólicos do DOcplex.
risk_expression = model.sum(
    float(
        covariance_values[i, j]
    )
    * variables[i]
    * variables[j]
    for i in range(n_assets)
    for j in range(n_assets)
)

return_expression = model.sum(
    float(
        assets_return_values[i]
    )
    * variables[i]
    for i in range(n_assets)
)

model.minimize(
    q_value
    * risk_expression
    - (1 - q_value)
    * return_expression
    + risk_free
)

model.add_constraint(
    model.sum(
        variables.tolist()
    )
    == budget,
    ctname="budget_exactly_four",
)

quad_model = from_docplex_mp(
    model=model
)

qubo_converter = (
    QuadraticProgramToQubo()
)

qubo = qubo_converter.convert(
    quad_model
)

ising, offset = qubo.to_ising()

print("QuadraticProgram:")
print(quad_model.prettyprint())

print()
print("QUBO:")
print(qubo.prettyprint())

print()
print("qubits do Ising:", ising.num_qubits)
print("offset:", offset)
print("termos de Pauli:", len(ising.paulis))


In [ ]:

# ============================================================
# 6. SOLUÇÃO EXATA DO HAMILTONIANO
# ============================================================

exact_solver = (
    NumPyMinimumEigensolver()
)

exact_result = (
    exact_solver
    .compute_minimum_eigenvalue(
        operator=ising
    )
)

exact_energy = float(
    np.real(
        exact_result.eigenvalue
        + offset
    )
)

exact_state_dict = (
    exact_result
    .eigenstate
    .to_dict()
)

exact_bitstring = str(
    max(
        exact_state_dict,
        key=lambda key: abs(
            exact_state_dict[key]
        ) ** 2,
    )
).replace(
    " ",
    "",
)

print("energia exata Ising + offset:", exact_energy)
print("bitstring exato do Ising:", exact_bitstring)
print("energia esperada do código original:", known_exact_energy)
print("bitstring esperado:", known_exact_bitstring)

energy_matches = np.isclose(
    exact_energy,
    known_exact_energy,
    atol=1e-10,
    rtol=0.0,
)

bitstring_matches = (
    exact_bitstring
    == known_exact_bitstring
)

print("energia coincide?", energy_matches)
print("bitstring coincide?", bitstring_matches)

if not energy_matches:
    raise RuntimeError(
        "A energia reconstruída não coincide com a energia "
        "do problema original. Verifique preços, retorno=sum(), "
        "q=0.5, risk_free=0.0475 e período dos dados."
    )

if not bitstring_matches:
    raise RuntimeError(
        "O bitstring reconstruído não coincide com 1001011000. "
        "Não execute o teste dos theta antes de corrigir "
        "a reconstrução do problema."
    )

EXACT_ENERGY = exact_energy
EXACT_BITSTRING = exact_bitstring



# Parte II — ansatz de Dicke parametrizado embutido no notebook

As três funções usadas no código original agora estão definidas diretamente neste arquivo:

```python
CY_parameterized(...)
CCY_parameterized(...)
DickeStateAnsatz(n, k)
```

Isso elimina completamente a dependência de:

```text
01_generate_dataset.ipynb
```

## Onde `DickeStateAnsatz` é chamada

No teste tradicional do código original:

```python
ansatz = DickeStateAnsatz(
    n=ising.num_qubits,
    k=4,
)
ansatz = ansatz.decompose()
```

Na geração das rodadas:

```python
ansatz_base = DickeStateAnsatz(
    n=ising.num_qubits,
    k=4,
).decompose()
```

Neste experimento será criado **somente `ansatz_base`**, uma única vez. Todas as estratégias — COBYLA completo, Top-1, Top-2, Top-4 e controles — usam exatamente esse mesmo circuito.

> Isso é importante porque os nomes simbólicos dos parâmetros incluem um sufixo aleatório. Recriar o ansatz no meio do experimento pode produzir novos nomes e alterar a correspondência entre a posição do vetor e a porta física. Por isso, a semente é fixada antes da construção e o circuito-base é congelado para toda a comparação.


In [ ]:

# ============================================================
# 7. PORTAS PARAMETRIZADAS E DickeStateAnsatz
#    IMPLEMENTAÇÃO EMBUTIDA — SEM DEPENDER DE OUTRO NOTEBOOK
# ============================================================

def CY_parameterized(i):
    """
    Cria a porta parametrizada chamada CY no código original.

    Parâmetro
    ---------
    i:
        Identificador usado somente para gerar um nome simbólico
        único para o ângulo da porta.

    Funcionamento
    -------------
    A porta interna é uma rotação controlada em Y:
        controle = qubit 1
        alvo     = qubit 0

    O valor numérico do ângulo só será inserido mais tarde por
    `assign_parameters`.
    """
    param = ParameterVector(
        name=f"x{i}",
        length=1,
    )

    qc = QuantumCircuit(2)

    qc.cry(
        param[0],
        1,
        0,
    )

    gate = qc.to_gate(
        label="CY"
    )

    return gate


def CCY_parameterized(i):
    """
    Cria a porta parametrizada chamada CCY no código original.

    Ela usa:
    1. uma rotação RY;
    2. uma porta Toffoli CCX;
    3. a rotação inversa;
    4. uma segunda CCX.

    O parâmetro continua simbólico até a chamada de
    `assign_parameters`.
    """
    param = ParameterVector(
        name=f"y{i}",
        length=1,
    )

    qc = QuantumCircuit(3)

    qc.ry(
        param[0],
        0,
    )

    qc.ccx(
        2,
        1,
        0,
    )

    qc.ry(
        -param[0],
        0,
    )

    qc.ccx(
        2,
        1,
        0,
    )

    gate = qc.to_gate(
        label="CCY"
    )

    return gate


def DickeStateAnsatz(n, k):
    """
    Constrói o ansatz parametrizado de Dicke usado no código original.

    Parâmetros
    ----------
    n:
        Número total de qubits.

    k:
        Peso de Hamming fixo. Neste problema:
            n = 10
            k = 4

    Ideia física/combinatória
    -------------------------
    Os últimos k qubits começam em |1>, formando um estado inicial
    com exatamente k excitações. As portas CY e CCY redistribuem
    amplitudes dentro do subespaço de peso de Hamming k.

    Consequência
    ------------
    A busca fica concentrada nas C(10,4)=210 soluções viáveis,
    em vez de explorar indiscriminadamente os 2^10 estados.
    """
    qr = QuantumRegister(
        n,
        "q",
    )

    qc = QuantumCircuit(
        qr
    )

    # --------------------------------------------------------
    # Inicialização:
    # aplica X nos últimos k qubits.
    #
    # Para n=10 e k=4, começa em:
    # |0000001111> na convenção lógica do registrador.
    # --------------------------------------------------------
    for i in range(k):
        qc.x(
            n - i - 1
        )

    aux = 1

    # Percorre os qubits do final para o início.
    for l in range(n)[::-1]:

        # Para cada l, percorre até k posições anteriores.
        for i in range(
            l - 1,
            l - 1 - k,
            -1,
        ):
            if i >= 0:

                # O código original cria um nome praticamente único
                # concatenando:
                #   l, i, contador auxiliar e inteiro aleatório.
                #
                # Esse nome identifica o parâmetro simbólico da porta.
                a = (
                    str(l)
                    + str(i)
                    + str(aux)
                    + str(
                        np.random.randint(
                            0,
                            1e8,
                        )
                    )
                )

                if i == l - 1:
                    # Caso vizinho:
                    # CX -> CY parametrizada -> CX
                    qc.cx(
                        qr[i],
                        qr[l],
                    )

                    qc.append(
                        CY_parameterized(a),
                        [
                            qr[i],
                            qr[l],
                        ],
                    )

                    qc.cx(
                        qr[i],
                        qr[l],
                    )

                else:
                    # Caso não vizinho:
                    # CX -> CCY parametrizada -> CX
                    #
                    # Mantém a mesma forma do código original,
                    # inclusive o uso de índices inteiros em qc.cx.
                    qc.cx(
                        i,
                        l,
                    )

                    qc.append(
                        CCY_parameterized(a),
                        [
                            qr[i],
                            qr[i + 1],
                            qr[l],
                        ],
                    )

                    qc.cx(
                        i,
                        l,
                    )

            # O contador avança em toda passagem do laço interno,
            # exatamente como na construção original.
            aux += 1

    return qc


# ------------------------------------------------------------
# REPRODUTIBILIDADE DA ORDEM DOS PARÂMETROS
# ------------------------------------------------------------
#
# Os identificadores das portas contêm np.random.randint(...).
# A semente é fixada imediatamente antes de criar o ansatz.
#
# IMPORTANTE:
# não execute novamente somente esta parte durante uma comparação
# já iniciada. Recrie o kernel e execute o notebook em ordem.
parameter_name_seed = random_seed

np.random.seed(
    parameter_name_seed
)

print(
    "CY_parameterized definida:",
    callable(CY_parameterized),
)

print(
    "CCY_parameterized definida:",
    callable(CCY_parameterized),
)

print(
    "DickeStateAnsatz definida:",
    callable(DickeStateAnsatz),
)


In [ ]:

# ============================================================
# 8. CHAMADA DO DickeStateAnsatz E CONGELAMENTO DO ANSATZ BASE
# ============================================================

# No teste tradicional do código original aparecia:
#
# ansatz = DickeStateAnsatz(
#     n=ising.num_qubits,
#     k=4,
# )
# ansatz = ansatz.decompose()
#
# Na geração das 10.000 rodadas aparecia:
#
# ansatz_base = DickeStateAnsatz(
#     n=ising.num_qubits,
#     k=4,
# ).decompose()
#
# Neste experimento usamos apenas `ansatz_base`.
# Ele é criado uma única vez e compartilhado por todos os métodos.

ansatz_base = DickeStateAnsatz(
    n=ising.num_qubits,
    k=budget,
).decompose()

n_parameters = int(
    ansatz_base.num_parameters
)

if n_parameters != 30:
    raise RuntimeError(
        "O ansatz deveria possuir 30 parâmetros para "
        f"n=10 e k=4, mas possui {n_parameters}."
    )

# O Qiskit associa uma sequência de 30 números à ordem abaixo.
# Essa tabela é essencial para interpretar:
# theta_0, theta_1, ..., theta_29.
parameter_order = list(
    ansatz_base.parameters
)

parameter_index_df = pd.DataFrame(
    {
        "theta_index": np.arange(
            n_parameters
        ),
        "parameter_name": [
            str(parameter)
            for parameter
            in parameter_order
        ],
    }
)

parameter_index_df.to_csv(
    output_dir
    / "theta_index_parameter_mapping.csv",
    index=False,
)

print(
    "qubits do ansatz:",
    ansatz_base.num_qubits,
)

print(
    "parâmetros simbólicos:",
    n_parameters,
)

print(
    "ansatz criado uma única vez e congelado para todo o experimento."
)

display(
    parameter_index_df
)



# Auditoria do mapeamento físico dos \(\theta\)

O objetivo desta auditoria é responder:

> O parâmetro chamado \(\theta_{17}\) no banco original atua sobre a mesma porta e os mesmos qubits que o índice 17 deste experimento?

Comparar apenas o nome simbólico não é suficiente, porque a função original acrescenta um número aleatório ao nome de cada parâmetro. A comparação correta usa uma **assinatura física**, formada por:

- posição da instrução no circuito;
- tipo da porta;
- qubits envolvidos;
- posição do parâmetro dentro da porta;
- sinal/coficiente com que o parâmetro aparece.

A auditoria possui três níveis:

1. **autoteste:** reconstrói o mesmo ansatz com duas sementes e verifica se os índices mudam;
2. **exportação do circuito atual:** salva o mapa físico deste experimento;
3. **comparação com o circuito original:** lê o mapa exportado no kernel que gerou o banco e traduz os índices originais para os índices atuais.


In [ ]:

# ============================================================
# AUDITORIA A. CONSTRUIR O MAPA FÍSICO DE CADA THETA
# ============================================================

import json


def _parameter_linear_coefficient(expression, parameter):
    """
    Obtém o coeficiente linear com que um parâmetro aparece
    em uma expressão de porta.

    Exemplos:
        theta      -> +1
        -theta     -> -1
        2 * theta  -> +2
    """
    try:
        value_zero = expression.bind({parameter: 0.0})
        value_one = expression.bind({parameter: 1.0})
        return float(value_one - value_zero)
    except Exception:
        try:
            value_zero = expression.assign(parameter, 0.0)
            value_one = expression.assign(parameter, 1.0)
            return float(value_one - value_zero)
        except Exception:
            return np.nan


def build_theta_physical_map(circuit):
    """
    Cria um mapa theta_index -> assinatura física.

    A assinatura não usa o nome aleatório do parâmetro.
    Ela descreve onde o parâmetro atua no circuito.
    """
    ordered_parameters = list(circuit.parameters)

    qubit_index = {
        qubit: index
        for index, qubit in enumerate(circuit.qubits)
    }

    uses = {
        parameter: []
        for parameter in ordered_parameters
    }

    for instruction_index, instruction in enumerate(circuit.data):
        operation = instruction.operation

        operation_qubits = [
            int(qubit_index[qubit])
            for qubit in instruction.qubits
        ]

        for parameter_slot, expression in enumerate(operation.params):
            expression_parameters = getattr(
                expression,
                "parameters",
                set(),
            )

            for parameter in expression_parameters:
                if parameter not in uses:
                    continue

                coefficient = _parameter_linear_coefficient(
                    expression,
                    parameter,
                )

                uses[parameter].append(
                    {
                        "instruction_index": int(instruction_index),
                        "operation": str(operation.name),
                        "qubits": operation_qubits,
                        "parameter_slot": int(parameter_slot),
                        "coefficient": (
                            None
                            if not np.isfinite(coefficient)
                            else round(float(coefficient), 12)
                        ),
                    }
                )

    rows = []

    for theta_index, parameter in enumerate(ordered_parameters):
        occurrences = uses[parameter]

        physical_signature = json.dumps(
            occurrences,
            sort_keys=True,
            separators=(",", ":"),
        )

        rows.append(
            {
                "theta_index": int(theta_index),
                "parameter_name": str(parameter),
                "n_occurrences": int(len(occurrences)),
                "physical_signature": physical_signature,
                "occurrences_readable": json.dumps(
                    occurrences,
                    indent=2,
                    ensure_ascii=False,
                ),
            }
        )

    mapping_df = pd.DataFrame(rows)

    if mapping_df["physical_signature"].duplicated().any():
        duplicated = mapping_df[
            mapping_df["physical_signature"].duplicated(keep=False)
        ]

        raise RuntimeError(
            "Foram encontradas assinaturas físicas repetidas. "
            "A correspondência não é unívoca.\n"
            + duplicated[
                [
                    "theta_index",
                    "parameter_name",
                    "physical_signature",
                ]
            ].to_string(index=False)
        )

    return mapping_df


current_theta_map = build_theta_physical_map(ansatz_base)

current_theta_map.to_csv(
    output_dir / "experiment_theta_physical_map.csv",
    index=False,
)

display(
    current_theta_map[
        [
            "theta_index",
            "parameter_name",
            "n_occurrences",
            "physical_signature",
        ]
    ]
)


In [ ]:

# ============================================================
# AUDITORIA B. AUTOTESTE COM DUAS SEMENTES
# ============================================================


def build_ansatz_with_seed(seed):
    """
    Reconstrói temporariamente o ansatz preservando
    o estado aleatório global do notebook.
    """
    numpy_random_state = np.random.get_state()

    try:
        np.random.seed(int(seed))
        temporary_ansatz = DickeStateAnsatz(
            n=ising.num_qubits,
            k=budget,
        ).decompose()
    finally:
        np.random.set_state(numpy_random_state)

    return temporary_ansatz


audit_ansatz_seed_a = build_ansatz_with_seed(random_seed)
audit_ansatz_seed_b = build_ansatz_with_seed(random_seed + 1001)

audit_map_seed_a = build_theta_physical_map(audit_ansatz_seed_a)
audit_map_seed_b = build_theta_physical_map(audit_ansatz_seed_b)

# A junção é feita pela assinatura física, e não pelo nome.
seed_comparison_df = audit_map_seed_a.merge(
    audit_map_seed_b,
    on="physical_signature",
    suffixes=("_seed_a", "_seed_b"),
    validate="one_to_one",
)

seed_comparison_df["same_theta_index"] = (
    seed_comparison_df["theta_index_seed_a"]
    == seed_comparison_df["theta_index_seed_b"]
)

same_index_fraction = float(
    seed_comparison_df["same_theta_index"].mean()
)

print(
    "parâmetros físicos comparados:",
    len(seed_comparison_df),
)

print(
    "fração que manteve o mesmo índice:",
    f"{same_index_fraction:.2%}",
)

display(
    seed_comparison_df[
        [
            "theta_index_seed_a",
            "parameter_name_seed_a",
            "theta_index_seed_b",
            "parameter_name_seed_b",
            "same_theta_index",
            "physical_signature",
        ]
    ].sort_values("theta_index_seed_a")
)

if same_index_fraction < 1.0:
    print(
        "\nRESULTADO DA AUDITORIA:"
        "\nOs nomes aleatórios alteram a associação "
        "índice -> porta física entre construções."
        "\nOs índices do banco original precisam ser traduzidos."
    )
else:
    print(
        "\nRESULTADO DA AUDITORIA:"
        "\nNesta versão do circuito, a ordem física permaneceu estável "
        "entre as duas sementes testadas."
    )



## Exportar o mapa do circuito que gerou as 10.000 execuções

A etapa decisiva deve ser executada **no notebook/kernel original**, onde está o `ansatz_base` utilizado na geração do banco.

Copie para o notebook original:

1. a célula `AUDITORIA A`, que define `build_theta_physical_map`;
2. depois execute:

```python
original_theta_map = build_theta_physical_map(
    ansatz_base
)

original_theta_map.to_csv(
    "../results/original_theta_physical_map.csv",
    index=False,
)

display(
    original_theta_map
)
```

O arquivo precisa ser salvo **antes de reiniciar o kernel original**, caso esse circuito ainda esteja em memória.

Se o kernel original já foi perdido e a semente/estado do NumPy usado naquela construção não foi registrado, não é possível provar retrospectivamente qual porta física correspondia a cada posição do vetor `best_parameters`. Nesse caso, o ranking deve ser reconstruído usando um ansatz determinístico.


In [ ]:

# ============================================================
# AUDITORIA C. COMPARAR ORIGINAL X EXPERIMENTO
# ============================================================

original_map_candidates = [
    Path("../results/original_theta_physical_map.csv"),
    Path("results/original_theta_physical_map.csv"),
    output_dir / "original_theta_physical_map.csv",
]

original_map_path = next(
    (
        candidate.resolve()
        for candidate in original_map_candidates
        if candidate.exists()
    ),
    None,
)

theta_translation_df = None

if original_map_path is None:
    print(
        "Mapa original ainda não localizado."
        "\nExporte-o no notebook/kernel que gerou o banco."
    )
else:
    original_theta_map = pd.read_csv(original_map_path)

    required_columns = {
        "theta_index",
        "physical_signature",
    }

    missing_columns = required_columns - set(original_theta_map.columns)

    if missing_columns:
        raise ValueError(
            "O mapa original não possui as colunas: "
            f"{sorted(missing_columns)}"
        )

    theta_translation_df = (
        original_theta_map[
            [
                "theta_index",
                "parameter_name",
                "physical_signature",
            ]
        ]
        .rename(
            columns={
                "theta_index": "theta_index_original",
                "parameter_name": "parameter_name_original",
            }
        )
        .merge(
            current_theta_map[
                [
                    "theta_index",
                    "parameter_name",
                    "physical_signature",
                ]
            ].rename(
                columns={
                    "theta_index": "theta_index_experiment",
                    "parameter_name": "parameter_name_experiment",
                }
            ),
            on="physical_signature",
            how="inner",
            validate="one_to_one",
        )
        .sort_values("theta_index_original")
        .reset_index(drop=True)
    )

    theta_translation_df["same_index"] = (
        theta_translation_df["theta_index_original"]
        == theta_translation_df["theta_index_experiment"]
    )

    if len(theta_translation_df) != n_parameters:
        raise RuntimeError(
            "Nem todas as 30 assinaturas físicas foram encontradas. "
            f"Foram associadas {len(theta_translation_df)}."
        )

    theta_translation_df.to_csv(
        output_dir / "original_to_experiment_theta_translation.csv",
        index=False,
    )

    print("mapa original:", original_map_path)
    print(
        "índices que permaneceram iguais:",
        int(theta_translation_df["same_index"].sum()),
        "de",
        n_parameters,
    )

    display(theta_translation_df)

    original_top_indices = [2, 3, 14, 17]

    translated_top_indices_df = (
        theta_translation_df[
            theta_translation_df["theta_index_original"].isin(
                original_top_indices
            )
        ][
            [
                "theta_index_original",
                "theta_index_experiment",
                "parameter_name_original",
                "parameter_name_experiment",
                "physical_signature",
            ]
        ]
        .sort_values("theta_index_original")
    )

    print("\nTradução dos índices prioritários:")
    display(translated_top_indices_df)



## O que `assign_parameters` faz

O circuito `ansatz_base` contém portas com parâmetros simbólicos. Por exemplo, internamente podem existir objetos equivalentes a:

```text
x... [0]
y... [0]
```

Depois que o COBYLA encontra valores numéricos, esta linha:

```python
ansatz_copy = (
    ansatz_base
    .copy()
    .assign_parameters(theta_completo)
)
```

substitui cada parâmetro simbólico pelo valor correspondente de `theta_completo`.

Ela **não otimiza**, **não calcula energia** e **não altera o Hamiltoniano**. Ela apenas cria a versão numérica do circuito para a simulação e a medição.

### Diferença crucial no experimento reduzido

No COBYLA completo:

```python
result_optimizer.x.shape == (30,)
```

No Top-4:

```python
result_optimizer.x.shape == (4,)
```

Portanto, isto estaria errado:

```python
# ERRADO no experimento Top-4:
ansatz_base.assign_parameters(
    result_optimizer.x
)
```

Antes, é obrigatório reconstruir:

\[
\theta_{\mathrm{completo}}
=
(\theta_0,\ldots,\theta_{29}),
\]

inserindo os quatro valores otimizados e preservando os outros 26 valores fixos.


In [ ]:

# ============================================================
# 9. SIMULADOR, ESTIMADOR E FUNÇÃO DE ENERGIA ORIGINAL
# ============================================================

simulator = AerSimulator(
    method="statevector",
    device="CPU",
)

estimator = BackendEstimatorV2(
    backend=simulator
)

# O callback é reiniciado antes de cada otimização.
callback_list = []


def scalarize_energy(
    value: Any,
) -> float:
    """
    Converte saídas NumPy escalares ou arrays de um elemento em float.
    """
    return float(
        np.real(
            np.asarray(
                value
            ).reshape(-1)[0]
        )
    )


def exp_val(
    x: np.ndarray,
) -> float:
    """
    Função objetivo usada pelo COBYLA.

    Esta implementação preserva a lógica do código original:
    1. copia o ansatz;
    2. decompõe o circuito;
    3. envia circuito, Ising e vetor theta ao Estimator;
    4. adiciona o offset;
    5. salva theta e energia no callback.
    """
    global callback_list

    x = np.asarray(
        x,
        dtype=float,
    ).reshape(-1)

    if x.size != n_parameters:
        raise ValueError(
            "exp_val sempre exige o vetor COMPLETO com "
            f"{n_parameters} parâmetros; recebeu {x.size}."
        )

    ansatz_copy = (
        ansatz_base
        .copy()
        .decompose()
    )

    estimator_result = estimator.run(
        pubs=[
            (
                ansatz_copy,
                ising,
                x,
            )
        ]
    ).result()

    cost_function_value = scalarize_energy(
        estimator_result[0].data.evs
    )

    original_energy = float(
        cost_function_value
        + offset
    )

    callback_list.append(
        (
            x.copy(),
            original_energy,
        )
    )

    return original_energy


print(
    "função exp_val pronta para",
    n_parameters,
    "parâmetros",
)


In [ ]:

# ============================================================
# 10. AUDITORIA EXPLÍCITA DO assign_parameters
# ============================================================

def assign_full_parameters(
    theta_full: np.ndarray,
) -> QuantumCircuit:
    """
    Converte o ansatz simbólico em circuito numérico.

    IMPORTANTE:
    - theta_full precisa possuir 30 valores;
    - inplace=False preserva ansatz_base para todas as rodadas;
    - a atribuição é posicional e segue parameter_order.
    """
    theta_full = np.asarray(
        theta_full,
        dtype=float,
    ).reshape(-1)

    if theta_full.size != n_parameters:
        raise ValueError(
            "assign_parameters exige o vetor completo: "
            f"esperados {n_parameters}, recebidos {theta_full.size}."
        )

    # RELAÇÃO EXATA COM A LINHA DA FOTO:
    #
    # ansatz_copy = (
    #     ansatz_base
    #     .copy()
    #     .assign_parameters(
    #         result_optimizer.x
    #     )
    # )
    #
    # No experimento reduzido usamos theta_full, pois
    # result_optimizer.x contém somente os parâmetros ativos.
    ansatz_numeric = (
        ansatz_base
        .copy()
        .assign_parameters(
            theta_full,
            inplace=False,
        )
    )

    if ansatz_numeric.num_parameters != 0:
        raise RuntimeError(
            "Ainda restaram parâmetros simbólicos após assign_parameters."
        )

    return ansatz_numeric


# Teste simples com um vetor de 30 valores.
theta_assignment_test = np.zeros(
    n_parameters,
    dtype=float,
)

assigned_test_circuit = (
    assign_full_parameters(
        theta_assignment_test
    )
)

print(
    "parâmetros restantes após atribuição:",
    assigned_test_circuit.num_parameters,
)

print(
    "ansatz_base continua simbólico:",
    ansatz_base.num_parameters,
)

print(
    "cópia atribuída é totalmente numérica:",
    assigned_test_circuit.num_parameters == 0,
)



# Parte III — COBYLA em subespaço ativo

Para um conjunto ativo \(S\), o COBYLA recebe somente:

\[
z=(\theta_j)_{j\in S}.
\]

A função objetivo reconstrói o vetor completo:

\[
\theta_j=
\begin{cases}
z_j, & j\in S,\\
\theta_j^{\mathrm{fixo}}, & j\notin S.
\end{cases}
\]

Então `exp_val` continua recebendo exatamente 30 parâmetros, como no código original.


In [ ]:

# ============================================================
# 11. RANKING E CONJUNTOS EXPERIMENTAIS
# ============================================================

# Ranking empírico do maior para o menor contraste
# na taxa do bitstring exato.
ranked_indices = [
    17, 2, 14, 3, 24, 4, 18, 1, 16, 20,
    8, 6, 0, 19, 15, 28, 5, 29, 7, 9,
    23, 25, 10, 21, 13, 11, 12, 22, 26, 27,
]

if sorted(
    ranked_indices
) != list(
    range(n_parameters)
):
    raise RuntimeError(
        "ranked_indices precisa conter 0...29 sem repetição."
    )

def translate_original_theta_indices(original_indices):
    """
    Traduz índices do circuito original para o circuito atual
    usando a assinatura física.

    Sem mapa original disponível, mantém os índices informados,
    mas registra que a correspondência ainda não foi comprovada.
    """
    original_indices = [int(index) for index in original_indices]

    if (
        "theta_translation_df" not in globals()
        or theta_translation_df is None
    ):
        warnings.warn(
            "O mapa físico original não foi carregado. "
            "Os índices serão usados sem tradução."
        )
        return sorted(original_indices)

    translation = dict(
        zip(
            theta_translation_df["theta_index_original"].astype(int),
            theta_translation_df["theta_index_experiment"].astype(int),
        )
    )

    missing = [
        index
        for index in original_indices
        if index not in translation
    ]

    if missing:
        raise RuntimeError(
            "Não foi possível traduzir os índices: "
            f"{missing}"
        )

    return sorted(translation[index] for index in original_indices)


# Estes são os índices encontrados na auditoria do BANCO ORIGINAL.
top1 = translate_original_theta_indices([17])
top2 = translate_original_theta_indices([14, 17])
top4 = translate_original_theta_indices([2, 3, 14, 17])

# Os controles também pertencem ao ranking original.
bottom4 = translate_original_theta_indices([12, 22, 26, 27])

random_pool = [
    index
    for index
    in range(n_parameters)
    if index not in top4
]

random4 = sorted(
    rng.choice(
        random_pool,
        size=4,
        replace=False,
    ).tolist()
)

experiments = {
    "full_30": list(
        range(n_parameters)
    ),
    "top1_theta17": top1,
    "top2_theta14_theta17": top2,
    "top4_theta02_03_14_17": top4,
    "bottom4_control": bottom4,
    "random4_control": random4,
}

display(
    pd.DataFrame(
        [
            {
                "method": method,
                "n_active": len(indices),
                "active_indices": indices,
            }
            for method, indices
            in experiments.items()
        ]
    )
)


In [ ]:

# ============================================================
# 12. PONTOS INICIAIS PAREADOS
# ============================================================

# O código original gera os theta do Dicke no intervalo:
#
# 0.5 * pi * random()
#
# Para uma comparação justa, cada restart possui UM vetor
# inicial completo. Todos os métodos usam exatamente esse
# mesmo vetor no mesmo restart.
initial_points = (
    0.5
    * np.pi
    * rng.random(
        size=(
            n_restarts,
            n_parameters,
        )
    )
)

initial_points_dict = {
    "Dicke_state": [
        row.copy()
        for row in initial_points
    ]
}

print(
    "matriz de pontos iniciais:",
    initial_points.shape,
)

print(
    "intervalo:",
    float(initial_points.min()),
    "até",
    float(initial_points.max()),
)


In [ ]:

# ============================================================
# 13. RECONSTRUÇÃO DO VETOR COMPLETO
# ============================================================

def reconstruct_full_theta(
    theta_active: np.ndarray,
    active_indices: Iterable[int],
    fixed_reference: np.ndarray,
) -> np.ndarray:
    """
    Insere os valores ativos no vetor fixo de 30 dimensões.

    Esta função é a ponte entre:
    - o COBYLA reduzido, que enxerga 1, 2, 4... variáveis;
    - o circuito original, que sempre exige 30 parâmetros.
    """
    active_indices = np.asarray(
        list(active_indices),
        dtype=int,
    )

    theta_active = np.asarray(
        theta_active,
        dtype=float,
    ).reshape(-1)

    fixed_reference = np.asarray(
        fixed_reference,
        dtype=float,
    ).reshape(-1)

    if fixed_reference.size != n_parameters:
        raise ValueError(
            "fixed_reference precisa possuir "
            f"{n_parameters} valores."
        )

    if theta_active.size != active_indices.size:
        raise ValueError(
            "Quantidade de valores ativos diferente "
            "da quantidade de índices ativos."
        )

    theta_full = (
        fixed_reference.copy()
    )

    theta_full[
        active_indices
    ] = theta_active

    return theta_full


In [ ]:

# ============================================================
# 14. MEDIÇÃO FINAL COM assign_parameters
# ============================================================

def sample_final_solution(
    theta_full: np.ndarray,
    seed_simulator: int,
) -> dict[str, Any]:
    """
    Liga os 30 theta ao circuito e executa 4096 shots.

    Esta é a etapa correspondente ao bloco mostrado nas fotos:
        ansatz_base.copy().assign_parameters(...)
        measure_all()
        simulator.run(...).get_counts()
    """
    theta_full = np.asarray(
        theta_full,
        dtype=float,
    ).reshape(-1)

    # AQUI ocorre o assign_parameters.
    # Ele recebe o vetor COMPLETO reconstruído.
    ansatz_copy = (
        assign_full_parameters(
            theta_full
        )
    )

    # A medição é adicionada somente à cópia numérica.
    # ansatz_base permanece simbólico e reutilizável.
    ansatz_copy.measure_all()

    simulator_start = perf_counter()

    counts = (
        simulator
        .run(
            ansatz_copy,
            shots=shots,
            seed_simulator=seed_simulator,
        )
        .result()
        .get_counts()
    )

    simulator_time = (
        perf_counter()
        - simulator_start
    )

    counts_clean = {
        str(key).replace(
            " ",
            "",
        ): int(value)
        for key, value
        in counts.items()
    }

    total_counts = int(
        sum(
            counts_clean.values()
        )
    )

    most_frequent_bitstring = max(
        counts_clean,
        key=counts_clean.get,
    )

    exact_count = counts_clean.get(
        EXACT_BITSTRING,
        0,
    )

    exact_probability = (
        exact_count
        / total_counts
        if total_counts > 0
        else np.nan
    )

    return {
        "counts": counts_clean,
        "total_counts": total_counts,
        "most_frequent_bitstring": most_frequent_bitstring,
        "probability_best_answer": exact_probability,
        "is_exact_dominant": (
            most_frequent_bitstring
            == EXACT_BITSTRING
        ),
        "time_spent_simulator": simulator_time,
    }


In [ ]:

# ============================================================
# 15. EXECUÇÃO DE UMA OTIMIZAÇÃO EM SUBESPAÇO ATIVO
# ============================================================

def run_active_cobyla(
    method: str,
    active_indices: Iterable[int],
    restart: int,
    theta_initial_full: np.ndarray,
    maxiter: int,
) -> dict[str, Any]:
    """
    Executa uma rodada completa:
    1. congela os theta fora do conjunto ativo;
    2. otimiza somente os theta ativos;
    3. reconstrói os 30 theta;
    4. usa assign_parameters;
    5. mede 4096 shots;
    6. salva energia, nfev, bitstring e tempos.
    """
    global callback_list

    active_indices = sorted(
        set(
            int(index)
            for index
            in active_indices
        )
    )

    theta_initial_full = np.asarray(
        theta_initial_full,
        dtype=float,
    ).reshape(-1)

    if theta_initial_full.size != n_parameters:
        raise ValueError(
            "theta_initial_full precisa possuir 30 valores."
        )

    # Os parâmetros fora de active_indices ficam exatamente
    # nos valores de theta_initial_full.
    fixed_reference = (
        theta_initial_full.copy()
    )

    x0_active = theta_initial_full[
        active_indices
    ].copy()

    callback_list = []

    def restricted_objective(
        theta_active,
    ):
        theta_full = reconstruct_full_theta(
            theta_active=theta_active,
            active_indices=active_indices,
            fixed_reference=fixed_reference,
        )

        # exp_val continua vendo 30 parâmetros.
        return exp_val(
            theta_full
        )

    optimizer = COBYLA(
        maxiter=int(maxiter)
    )

    optimizer_start = perf_counter()

    try:
        result_optimizer = optimizer.minimize(
            fun=restricted_objective,
            x0=x0_active,
        )
    except Exception as exc:
        return {
            "method": method,
            "restart": restart,
            "status": "failed",
            "error": (
                f"{type(exc).__name__}: {exc}"
            ),
        }

    optimizer_time = (
        perf_counter()
        - optimizer_start
    )

    # result_optimizer.x possui somente n_active valores.
    theta_final_full = reconstruct_full_theta(
        theta_active=result_optimizer.x,
        active_indices=active_indices,
        fixed_reference=fixed_reference,
    )

    # Número de avaliações reais da energia.
    nfev_callback = len(
        callback_list
    )

    nfev_result = getattr(
        result_optimizer,
        "nfev",
        np.nan,
    )

    # assign_parameters e shots acontecem aqui.
    sampling = sample_final_solution(
        theta_full=theta_final_full,
        seed_simulator=(
            random_seed
            + 10_000
            * restart
            + len(active_indices)
        ),
    )

    final_energy = float(
        result_optimizer.fun
    )

    energy_gap = abs(
        final_energy
        - EXACT_ENERGY
    )

    return {
        "method": method,
        "restart": restart,
        "status": "ok",
        "n_active": len(active_indices),
        "active_indices": active_indices,
        "theta_initial_full": theta_initial_full,
        "theta_final_full": theta_final_full,
        "theta_active_final": np.asarray(
            result_optimizer.x,
            dtype=float,
        ),
        "objective_function_value": final_energy,
        "energy_gap": energy_gap,
        "is_near_optimal": (
            energy_gap
            < near_optimal_threshold
        ),
        "nfev_callback": nfev_callback,
        "nfev_result": nfev_result,
        "time_spent_optimizer": optimizer_time,
        "callback_list": [
            (
                np.asarray(
                    theta,
                    dtype=float,
                ),
                float(energy),
            )
            for theta, energy
            in callback_list
        ],
        **sampling,
    }


In [ ]:

# ============================================================
# 16. EXECUTAR A SUÍTE PRINCIPAL COM CHECKPOINT
# ============================================================

runs_pickle_path = (
    output_dir
    / "active_subspace_runs.pkl"
)

if runs_pickle_path.exists():
    all_runs = pd.read_pickle(
        runs_pickle_path
    )

    if not isinstance(
        all_runs,
        list,
    ):
        all_runs = []
else:
    all_runs = []

completed_keys = {
    (
        run.get("method"),
        int(run.get("restart")),
    )
    for run in all_runs
    if run.get("status") == "ok"
}

for restart in range(
    n_restarts
):
    theta_initial_full = (
        initial_points[
            restart
        ].copy()
    )

    for method, active_indices in experiments.items():
        key = (
            method,
            restart,
        )

        if key in completed_keys:
            print(
                "pulando checkpoint:",
                key,
            )
            continue

        print()
        print(
            "=" * 78
        )
        print(
            method,
            "| restart",
            restart,
            "| ativos",
            active_indices,
        )
        print(
            "=" * 78
        )

        run = run_active_cobyla(
            method=method,
            active_indices=active_indices,
            restart=restart,
            theta_initial_full=theta_initial_full,
            maxiter=maxiter_main,
        )

        all_runs.append(
            run
        )

        pd.to_pickle(
            all_runs,
            runs_pickle_path,
        )

        if run["status"] == "ok":
            print(
                "nfev:",
                run["nfev_callback"],
                "| gap:",
                f"{run['energy_gap']:.6e}",
                "| P(exato):",
                f"{run['probability_best_answer']:.4f}",
                "| dominante:",
                run["most_frequent_bitstring"],
            )
        else:
            print(
                "falhou:",
                run["error"],
            )


In [ ]:

# ============================================================
# 17. TOP-4 SEGUIDO DE REFINAMENTO COMPLETO
# ============================================================

if run_two_stage:

    for restart in range(
        n_restarts
    ):
        method = (
            "top4_then_full_refinement"
        )

        key = (
            method,
            restart,
        )

        if key in completed_keys:
            print(
                "pulando checkpoint:",
                key,
            )
            continue

        theta_initial_full = (
            initial_points[
                restart
            ].copy()
        )

        stage1 = run_active_cobyla(
            method="two_stage_top4",
            active_indices=top4,
            restart=restart,
            theta_initial_full=theta_initial_full,
            maxiter=maxiter_main,
        )

        if stage1["status"] != "ok":
            all_runs.append(
                {
                    "method": method,
                    "restart": restart,
                    "status": "failed",
                    "error": (
                        "falha no estágio Top-4: "
                        + stage1.get(
                            "error",
                            "erro desconhecido",
                        )
                    ),
                }
            )
            continue

        # O refinamento completo parte da solução Top-4.
        stage2 = run_active_cobyla(
            method=method,
            active_indices=list(
                range(n_parameters)
            ),
            restart=restart,
            theta_initial_full=stage1[
                "theta_final_full"
            ],
            maxiter=maxiter_refinement,
        )

        if stage2["status"] == "ok":
            stage2[
                "nfev_callback"
            ] += stage1[
                "nfev_callback"
            ]

            stage2[
                "time_spent_optimizer"
            ] += stage1[
                "time_spent_optimizer"
            ]

            stage2[
                "callback_list"
            ] = (
                stage1[
                    "callback_list"
                ]
                + stage2[
                    "callback_list"
                ]
            )

            stage2[
                "stage1_top4_energy"
            ] = stage1[
                "objective_function_value"
            ]

        all_runs.append(
            stage2
        )

        pd.to_pickle(
            all_runs,
            runs_pickle_path,
        )


In [ ]:

# ============================================================
# 18. CONVERTER OS RESULTADOS EM TABELAS
# ============================================================

result_rows = []

for run in all_runs:
    row = {
        "method": run.get(
            "method"
        ),
        "restart": run.get(
            "restart"
        ),
        "status": run.get(
            "status"
        ),
        "error": run.get(
            "error"
        ),
    }

    if run.get(
        "status"
    ) == "ok":
        row.update(
            {
                "n_active": run[
                    "n_active"
                ],
                "active_indices": json.dumps(
                    run[
                        "active_indices"
                    ]
                ),
                "objective_function_value": run[
                    "objective_function_value"
                ],
                "energy_gap": run[
                    "energy_gap"
                ],
                "nfev": run[
                    "nfev_callback"
                ],
                "nfev_result": run[
                    "nfev_result"
                ],
                "time_spent_optimizer": run[
                    "time_spent_optimizer"
                ],
                "time_spent_simulator": run[
                    "time_spent_simulator"
                ],
                "probability_best_answer": run[
                    "probability_best_answer"
                ],
                "most_frequent_bitstring": run[
                    "most_frequent_bitstring"
                ],
                "is_exact_dominant": run[
                    "is_exact_dominant"
                ],
                "is_near_optimal": run[
                    "is_near_optimal"
                ],
            }
        )

    result_rows.append(
        row
    )

results_df = pd.DataFrame(
    result_rows
)

ok_df = results_df[
    results_df[
        "status"
    ]
    == "ok"
].copy()

summary_df = (
    ok_df
    .groupby(
        [
            "method",
            "n_active",
        ],
        as_index=False,
    )
    .agg(
        runs=(
            "restart",
            "count",
        ),
        nfev_median=(
            "nfev",
            "median",
        ),
        nfev_mean=(
            "nfev",
            "mean",
        ),
        optimizer_time_median=(
            "time_spent_optimizer",
            "median",
        ),
        simulator_time_median=(
            "time_spent_simulator",
            "median",
        ),
        gap_median=(
            "energy_gap",
            "median",
        ),
        gap_mean=(
            "energy_gap",
            "mean",
        ),
        p_exact_median=(
            "probability_best_answer",
            "median",
        ),
        exact_dominant_rate=(
            "is_exact_dominant",
            "mean",
        ),
        near_optimal_rate=(
            "is_near_optimal",
            "mean",
        ),
    )
)

summary_df[
    "exact_dominant_rate_pct"
] = (
    100
    * summary_df[
        "exact_dominant_rate"
    ]
)

summary_df[
    "near_optimal_rate_pct"
] = (
    100
    * summary_df[
        "near_optimal_rate"
    ]
)

results_df.to_csv(
    output_dir
    / "active_subspace_runs.csv",
    index=False,
)

summary_df.to_csv(
    output_dir
    / "active_subspace_summary.csv",
    index=False,
)

display(
    results_df.sort_values(
        [
            "method",
            "restart",
        ]
    )
)

display(
    summary_df.sort_values(
        [
            "gap_median",
            "nfev_median",
        ]
    )
)


In [ ]:

# ============================================================
# 19. CURVA DE DIMENSÃO ATIVA
# ============================================================

dimension_rows = []

if run_dimension_curve:

    active_dimensions = [
        1,
        2,
        4,
        6,
        8,
        12,
        16,
        30,
    ]

    dimension_checkpoint = (
        output_dir
        / "dimension_runs.pkl"
    )

    if dimension_checkpoint.exists():
        dimension_runs = pd.read_pickle(
            dimension_checkpoint
        )
    else:
        dimension_runs = []

    dimension_completed = {
        (
            int(run["dimension"]),
            int(run["restart"]),
        )
        for run in dimension_runs
        if run.get("status") == "ok"
    }

    for dimension in active_dimensions:
        active_indices = ranked_indices[
            :dimension
        ]

        for restart in range(
            n_restarts
        ):
            key = (
                dimension,
                restart,
            )

            if key in dimension_completed:
                continue

            run = run_active_cobyla(
                method=f"top_{dimension}",
                active_indices=active_indices,
                restart=restart,
                theta_initial_full=initial_points[
                    restart
                ],
                maxiter=maxiter_main,
            )

            run[
                "dimension"
            ] = dimension

            dimension_runs.append(
                run
            )

            pd.to_pickle(
                dimension_runs,
                dimension_checkpoint,
            )

    for run in dimension_runs:
        if run.get(
            "status"
        ) != "ok":
            continue

        dimension_rows.append(
            {
                "dimension": run[
                    "dimension"
                ],
                "restart": run[
                    "restart"
                ],
                "nfev": run[
                    "nfev_callback"
                ],
                "energy_gap": run[
                    "energy_gap"
                ],
                "probability_best_answer": run[
                    "probability_best_answer"
                ],
                "is_exact_dominant": run[
                    "is_exact_dominant"
                ],
            }
        )

dimension_df = pd.DataFrame(
    dimension_rows
)

if not dimension_df.empty:
    dimension_summary_df = (
        dimension_df
        .groupby(
            "dimension",
            as_index=False,
        )
        .agg(
            nfev_median=(
                "nfev",
                "median",
            ),
            gap_median=(
                "energy_gap",
                "median",
            ),
            p_exact_median=(
                "probability_best_answer",
                "median",
            ),
            exact_rate=(
                "is_exact_dominant",
                "mean",
            ),
        )
    )

    dimension_df.to_csv(
        output_dir
        / "dimension_runs.csv",
        index=False,
    )

    dimension_summary_df.to_csv(
        output_dir
        / "dimension_summary.csv",
        index=False,
    )

    display(
        dimension_summary_df
    )


In [ ]:

# ============================================================
# 20. GRÁFICOS
# ============================================================

if not summary_df.empty:

    # --------------------------------------------------------
    # Avaliações da função objetivo
    # --------------------------------------------------------
    plot_df = summary_df.sort_values(
        "nfev_median"
    )

    fig, ax = plt.subplots(
        figsize=(12, 6)
    )

    bars = ax.barh(
        plot_df[
            "method"
        ],
        plot_df[
            "nfev_median"
        ],
    )

    for bar, value in zip(
        bars,
        plot_df[
            "nfev_median"
        ],
    ):
        ax.text(
            bar.get_width(),
            bar.get_y()
            + bar.get_height()
            / 2,
            f" {value:.0f}",
            va="center",
        )

    ax.set_xlabel(
        "Mediana de avaliações da energia"
    )
    ax.set_title(
        "Custo do COBYLA por estratégia"
    )
    ax.grid(
        axis="x",
        alpha=0.25,
    )

    fig.tight_layout()
    fig.savefig(
        output_dir
        / "01_nfev_por_metodo.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()

    # --------------------------------------------------------
    # Gap energético
    # --------------------------------------------------------
    plot_df = summary_df.sort_values(
        "gap_median"
    )

    fig, ax = plt.subplots(
        figsize=(12, 6)
    )

    ax.barh(
        plot_df[
            "method"
        ],
        plot_df[
            "gap_median"
        ],
    )

    ax.axvline(
        near_optimal_threshold,
        linestyle="--",
        linewidth=1.5,
        label="limite quase ótimo",
    )

    ax.set_xscale(
        "log"
    )
    ax.set_xlabel(
        "Gap energético mediano"
    )
    ax.set_title(
        "Qualidade energética por estratégia"
    )
    ax.grid(
        axis="x",
        alpha=0.25,
    )
    ax.legend()

    fig.tight_layout()
    fig.savefig(
        output_dir
        / "02_gap_por_metodo.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()

    # --------------------------------------------------------
    # Taxa de bitstring exato dominante
    # --------------------------------------------------------
    plot_df = summary_df.sort_values(
        "exact_dominant_rate_pct"
    )

    fig, ax = plt.subplots(
        figsize=(12, 6)
    )

    bars = ax.barh(
        plot_df[
            "method"
        ],
        plot_df[
            "exact_dominant_rate_pct"
        ],
    )

    for bar, value in zip(
        bars,
        plot_df[
            "exact_dominant_rate_pct"
        ],
    ):
        ax.text(
            bar.get_width(),
            bar.get_y()
            + bar.get_height()
            / 2,
            f" {value:.1f}%",
            va="center",
        )

    ax.set_xlim(
        0,
        105,
    )
    ax.set_xlabel(
        "Execuções com bitstring exato dominante (%)"
    )
    ax.set_title(
        "Recuperação de 1001011000"
    )
    ax.grid(
        axis="x",
        alpha=0.25,
    )

    fig.tight_layout()
    fig.savefig(
        output_dir
        / "03_taxa_bitstring_exato.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()


if (
    "dimension_summary_df"
    in globals()
    and not dimension_summary_df.empty
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        dimension_summary_df[
            "dimension"
        ],
        dimension_summary_df[
            "gap_median"
        ],
        marker="o",
        linewidth=2,
    )

    ax.axhline(
        near_optimal_threshold,
        linestyle="--",
        linewidth=1.5,
    )

    ax.set_yscale(
        "log"
    )
    ax.set_xlabel(
        "Quantidade de theta ativos"
    )
    ax.set_ylabel(
        "Gap energético mediano"
    )
    ax.set_title(
        "Menor dimensão ativa que preserva a solução"
    )
    ax.grid(
        alpha=0.25,
    )

    fig.tight_layout()
    fig.savefig(
        output_dir
        / "04_curva_dimensao_ativa.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()


In [ ]:

# ============================================================
# 21. DIAGNÓSTICO AUTOMÁTICO DE VIABILIDADE
# ============================================================

def get_summary_row(
    method,
):
    subset = summary_df[
        summary_df[
            "method"
        ]
        == method
    ]

    if subset.empty:
        return None

    return subset.iloc[0]


full_row = get_summary_row(
    "full_30"
)

top4_row = get_summary_row(
    "top4_theta02_03_14_17"
)

bottom4_row = get_summary_row(
    "bottom4_control"
)

random4_row = get_summary_row(
    "random4_control"
)

diagnostic = {
    "status": "insufficient_results",
}

if (
    full_row is not None
    and top4_row is not None
):
    nfev_reduction = (
        1.0
        - top4_row[
            "nfev_median"
        ]
        / full_row[
            "nfev_median"
        ]
    )

    gap_ratio = (
        top4_row[
            "gap_median"
        ]
        / max(
            full_row[
                "gap_median"
            ],
            1e-15,
        )
    )

    exact_rate_difference_pp = (
        100
        * (
            top4_row[
                "exact_dominant_rate"
            ]
            - full_row[
                "exact_dominant_rate"
            ]
        )
    )

    top4_beats_controls = True

    for control_row in [
        bottom4_row,
        random4_row,
    ]:
        if control_row is None:
            continue

        top4_beats_controls &= (
            top4_row[
                "gap_median"
            ]
            <= control_row[
                "gap_median"
            ]
        )

    diagnostic = {
        "status": "evaluated",
        "nfev_reduction_fraction": float(
            nfev_reduction
        ),
        "nfev_reduction_pct": float(
            100
            * nfev_reduction
        ),
        "gap_ratio_top4_over_full": float(
            gap_ratio
        ),
        "exact_rate_difference_pp": float(
            exact_rate_difference_pp
        ),
        "top4_beats_controls_on_gap": bool(
            top4_beats_controls
        ),
        "provisional_viability": bool(
            nfev_reduction > 0
            and gap_ratio <= 2.0
            and exact_rate_difference_pp >= -10.0
            and top4_beats_controls
        ),
    }

print(
    json.dumps(
        diagnostic,
        indent=2,
        ensure_ascii=False,
    )
)

with open(
    output_dir
    / "viability_diagnostic.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        diagnostic,
        file,
        indent=2,
        ensure_ascii=False,
    )



# Interpretação

## Resultado favorável

O Top-4 é útil quando apresenta:

\[
N_{\mathrm{fev}}^{\mathrm{Top4}}
<
N_{\mathrm{fev}}^{\mathrm{Full}},
\]

mantendo:

\[
\Delta E_{\mathrm{Top4}}
\approx
\Delta E_{\mathrm{Full}},
\]

e:

\[
P_{\mathrm{Top4}}(x^\star)
\approx
P_{\mathrm{Full}}(x^\star).
\]

Além disso, `bottom4_control` e `random4_control` precisam apresentar desempenho inferior. Caso contrário, o ganho pode vir apenas da redução dimensional, e não da seleção dos parâmetros sensíveis.

## Resultado intermediário

Se o Top-4 sozinho não alcançar a solução, mas:

```text
top4_then_full_refinement
```

usar menos avaliações do que `full_30` e recuperar a qualidade, então os parâmetros sensíveis funcionam como etapa de busca principal, enquanto os demais fazem o ajuste fino.

## Resultado negativo

Se apenas o COBYLA completo funcionar, os theta selecionados podem ser sensíveis, porém não suficientes. Isso indicaria forte interação e compensação entre os parâmetros.

## Limite científico

A primeira conclusão será válida para:

- este CSV;
- este período;
- esta matriz de covariância;
- \(q=0.5\);
- \(r_f=0.0475\);
- \(n=10\);
- \(k=4\);
- este Hamiltoniano;
- este ansatz;
- esta convenção de ordem dos parâmetros.

A generalização exige reconstruir o ranking em novos problemas, e não transferir automaticamente os índices \(\theta_2,\theta_3,\theta_{14},\theta_{17}\).



# Parte IV — fundo quase ótimo e liberação progressiva

A auditoria anterior mostrou que a associação entre índice e posição física permaneceu estável em 100% dos parâmetros. Agora são executados dois testes adicionais.

## Experimento A — suficiência local sobre fundo quase ótimo

Para cada repetição:

1. selecionar do banco uma solução quase ótima completa;
2. manter 26 parâmetros nos valores dessa solução;
3. retirar somente o Top-4 de seus valores bons;
4. recolocar o Top-4 em valores iniciais aleatórios;
5. otimizar apenas o Top-4;
6. verificar quanto da energia e do bitstring exato foi recuperado;
7. repetir o mesmo procedimento com o Bottom-4 como controle.

Esse teste não pergunta se quatro parâmetros constroem toda a solução do zero. Ele pergunta:

> Dado um fundo paramétrico adequado, o Top-4 consegue recuperar localmente a solução?

## Experimento B — liberação progressiva

A otimização começa com poucos parâmetros e amplia o conjunto ativo somente por estágios:

\[
1\rightarrow2\rightarrow4\rightarrow6\rightarrow8
\rightarrow12\rightarrow16\rightarrow30.
\]

O orçamento total de avaliações é mantido igual ao COBYLA completo. Assim, a comparação não favorece o método progressivo por permitir mais chamadas da função objetivo.


In [ ]:

# ============================================================
# 22. CONFIGURAÇÃO DOS NOVOS TESTES
# ============================================================

# Primeira execução:
#     followup_robust_mode = False  -> 3 repetições
#
# Resultado estatístico:
#     followup_robust_mode = True   -> 30 repetições
followup_robust_mode = False

followup_n_restarts = (
    30
    if followup_robust_mode
    else 3
)

# Mesmo orçamento máximo usado no gráfico atual do COBYLA completo.
followup_total_budget = 250

# No teste local, Top-4 e Bottom-4 recebem exatamente
# o mesmo orçamento.
local_active_budget = 80

run_near_optimal_background_test = True
run_progressive_release_test = True

# Definição de "quase ótimo" já usada no restante do notebook.
local_background_gap_threshold = (
    near_optimal_threshold
)

# Dá preferência a vetores quase ótimos cujo bitstring dominante
# no banco também foi o exato.
prefer_exact_dominant_backgrounds = True

followup_output_dir = (
    output_dir
    / "followup_local_progressive"
)

followup_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# Pontos iniciais pareados específicos desta etapa.
# O vetor do restart r é usado tanto pelo COBYLA completo
# quanto pelo método progressivo.
followup_rng = np.random.default_rng(
    random_seed + 2026
)

followup_initial_points = (
    0.5
    * np.pi
    * followup_rng.random(
        size=(
            followup_n_restarts,
            n_parameters,
        )
    )
)

print(
    "modo robusto:",
    followup_robust_mode,
)

print(
    "repetições:",
    followup_n_restarts,
)

print(
    "orçamento total por método:",
    followup_total_budget,
)

print(
    "orçamento do teste local:",
    local_active_budget,
)


In [ ]:

# ============================================================
# 23. CARREGAR O BANCO DAS 10.000 EXECUÇÕES
# ============================================================

import ast


# Ajuste manualmente somente quando o arquivo estiver em outro local.
manual_theta_bank_path = None

theta_bank_candidates = [
    Path("../results/merged.pkl"),
    Path("results/merged.pkl"),
    Path("../data/processed/merged.pkl"),
    Path("data/processed/merged.pkl"),
    Path("../data/merged.pkl"),
    Path("merged.pkl"),
]

if manual_theta_bank_path is not None:
    theta_bank_path = Path(
        manual_theta_bank_path
    ).resolve()
else:
    theta_bank_path = next(
        (
            candidate.resolve()
            for candidate
            in theta_bank_candidates
            if candidate.exists()
        ),
        None,
    )


def parse_theta_vector(value):
    """
    Converte a célula best_parameters em vetor NumPy.
    Funciona para ndarray, lista, tupla e string de lista.
    """
    if isinstance(
        value,
        np.ndarray,
    ):
        vector = value

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        vector = np.asarray(
            value
        )

    elif isinstance(
        value,
        str,
    ):
        vector = np.asarray(
            ast.literal_eval(
                value
            )
        )

    else:
        vector = np.asarray(
            value
        )

    return np.asarray(
        vector,
        dtype=float,
    ).reshape(-1)


theta_bank_df = None
theta_bank_matrix = None

if run_near_optimal_background_test:

    if theta_bank_path is None:
        raise FileNotFoundError(
            "merged.pkl não foi localizado. "
            "Defina manual_theta_bank_path na célula 23."
        )

    theta_bank_df = pd.read_pickle(
        theta_bank_path
    )

    required_bank_columns = {
        "best_parameters",
        "objective_function_value",
    }

    missing_bank_columns = (
        required_bank_columns
        - set(
            theta_bank_df.columns
        )
    )

    if missing_bank_columns:
        raise KeyError(
            "O banco não possui as colunas: "
            f"{sorted(missing_bank_columns)}"
        )

    parsed_theta = theta_bank_df[
        "best_parameters"
    ].map(
        parse_theta_vector
    )

    valid_theta_mask = parsed_theta.map(
        lambda vector: (
            vector.size
            == n_parameters
            and np.all(
                np.isfinite(
                    vector
                )
            )
        )
    )

    theta_bank_df = theta_bank_df.loc[
        valid_theta_mask
    ].copy()

    theta_bank_matrix = np.vstack(
        parsed_theta.loc[
            valid_theta_mask
        ].to_numpy()
    )

    theta_bank_df[
        "energy_numeric"
    ] = pd.to_numeric(
        theta_bank_df[
            "objective_function_value"
        ],
        errors="coerce",
    )

    theta_bank_df[
        "energy_gap_from_exact"
    ] = np.abs(
        theta_bank_df[
            "energy_numeric"
        ]
        - EXACT_ENERGY
    )

    if (
        "most_frequent_bitstring"
        in theta_bank_df.columns
    ):
        theta_bank_df[
            "bitstring_clean"
        ] = (
            theta_bank_df[
                "most_frequent_bitstring"
            ]
            .astype(str)
            .str.replace(
                " ",
                "",
                regex=False,
            )
        )
    else:
        theta_bank_df[
            "bitstring_clean"
        ] = ""

    theta_bank_df[
        "theta_bank_position"
    ] = np.arange(
        len(
            theta_bank_df
        )
    )

    print(
        "banco localizado:",
        theta_bank_path,
    )

    print(
        "vetores válidos:",
        theta_bank_matrix.shape,
    )

    print(
        "vetores com gap <",
        local_background_gap_threshold,
        ":",
        int(
            (
                theta_bank_df[
                    "energy_gap_from_exact"
                ]
                < local_background_gap_threshold
            ).sum()
        ),
    )


In [ ]:

# ============================================================
# 24. SELECIONAR FUNDOS QUASE ÓTIMOS
# ============================================================

near_backgrounds_df = None
near_background_matrix = None

if run_near_optimal_background_test:

    finite_mask = np.isfinite(
        theta_bank_df[
            "energy_numeric"
        ].to_numpy()
    )

    near_mask = (
        finite_mask
        & (
            theta_bank_df[
                "energy_gap_from_exact"
            ].to_numpy()
            < local_background_gap_threshold
        )
    )

    exact_near_mask = (
        near_mask
        & (
            theta_bank_df[
                "bitstring_clean"
            ].to_numpy()
            == EXACT_BITSTRING
        )
    )

    if (
        prefer_exact_dominant_backgrounds
        and int(
            exact_near_mask.sum()
        )
        >= followup_n_restarts
    ):
        eligible_mask = (
            exact_near_mask
        )
        background_selection_rule = (
            "gap quase ótimo + bitstring exato dominante"
        )
    else:
        eligible_mask = near_mask
        background_selection_rule = (
            "gap quase ótimo"
        )

    eligible_positions = np.flatnonzero(
        eligible_mask
    )

    if (
        eligible_positions.size
        < followup_n_restarts
    ):
        raise RuntimeError(
            "Não existem fundos quase ótimos suficientes. "
            f"Necessários: {followup_n_restarts}; "
            f"disponíveis: {eligible_positions.size}."
        )

    selected_positions = followup_rng.choice(
        eligible_positions,
        size=followup_n_restarts,
        replace=False,
    )

    near_backgrounds_df = (
        theta_bank_df
        .iloc[
            selected_positions
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    near_background_matrix = (
        theta_bank_matrix[
            selected_positions
        ].copy()
    )

    near_backgrounds_df[
        "background_id"
    ] = np.arange(
        followup_n_restarts
    )

    near_backgrounds_df.to_csv(
        followup_output_dir
        / "selected_near_optimal_backgrounds.csv",
        index=False,
    )

    print(
        "regra de seleção:",
        background_selection_rule,
    )

    print(
        "fundos selecionados:",
        near_background_matrix.shape,
    )

    display(
        near_backgrounds_df[
            [
                "background_id",
                "energy_numeric",
                "energy_gap_from_exact",
                "bitstring_clean",
            ]
        ]
    )


In [ ]:

# ============================================================
# 25. AVALIAR UM VETOR SEM OTIMIZAÇÃO
# ============================================================

def evaluate_without_optimization(
    method,
    restart,
    theta_full,
    seed_simulator,
    background_id=None,
):
    """
    Mede um vetor completo antes de chamar o COBYLA.

    Isso permite observar separadamente:
    - a qualidade do fundo quase ótimo;
    - o dano produzido ao retirar o Top-4;
    - a recuperação após reotimizar o Top-4.
    """
    global callback_list

    theta_full = np.asarray(
        theta_full,
        dtype=float,
    ).reshape(-1)

    callback_list = []

    energy = float(
        exp_val(
            theta_full
        )
    )

    sampling = sample_final_solution(
        theta_full=theta_full,
        seed_simulator=int(
            seed_simulator
        ),
    )

    return {
        "method": method,
        "restart": int(
            restart
        ),
        "background_id": background_id,
        "status": "ok",
        "n_active": 0,
        "active_indices": [],
        "theta_initial_full": theta_full.copy(),
        "theta_final_full": theta_full.copy(),
        "objective_function_value": energy,
        "energy_gap": abs(
            energy
            - EXACT_ENERGY
        ),
        "is_near_optimal": (
            abs(
                energy
                - EXACT_ENERGY
            )
            < near_optimal_threshold
        ),
        "nfev_callback": 1,
        "nfev_result": 0,
        "time_spent_optimizer": 0.0,
        "callback_list": [
            (
                theta.copy(),
                float(
                    callback_energy
                ),
            )
            for theta, callback_energy
            in callback_list
        ],
        **sampling,
    }


In [ ]:

# ============================================================
# 26. EXPERIMENTO A — FUNDO QUASE ÓTIMO
# ============================================================

local_checkpoint_path = (
    followup_output_dir
    / "near_background_local_runs.pkl"
)

if (
    run_near_optimal_background_test
    and local_checkpoint_path.exists()
):
    local_runs = pd.read_pickle(
        local_checkpoint_path
    )

    if not isinstance(
        local_runs,
        list,
    ):
        local_runs = []
else:
    local_runs = []


def local_run_key(
    method,
    restart,
):
    return (
        str(
            method
        ),
        int(
            restart
        ),
    )


local_completed = {
    local_run_key(
        run.get(
            "method"
        ),
        run.get(
            "restart"
        ),
    )
    for run in local_runs
    if run.get(
        "status"
    ) == "ok"
}


def save_local_run(
    run,
):
    key = local_run_key(
        run[
            "method"
        ],
        run[
            "restart"
        ],
    )

    if key not in local_completed:
        local_runs.append(
            run
        )

        local_completed.add(
            key
        )

        pd.to_pickle(
            local_runs,
            local_checkpoint_path,
        )


if run_near_optimal_background_test:

    for restart in range(
        followup_n_restarts
    ):
        good_background = (
            near_background_matrix[
                restart
            ].copy()
        )

        random_start = (
            followup_initial_points[
                restart
            ].copy()
        )

        background_id = int(
            near_backgrounds_df.loc[
                restart,
                "background_id",
            ]
        )

        # ----------------------------------------------------
        # A1. Fundo quase ótimo intacto
        # ----------------------------------------------------
        method = (
            "near_good_background"
        )

        if local_run_key(
            method,
            restart,
        ) not in local_completed:
            run = evaluate_without_optimization(
                method=method,
                restart=restart,
                theta_full=good_background,
                seed_simulator=(
                    random_seed
                    + 100_000
                    + restart
                ),
                background_id=background_id,
            )

            save_local_run(
                run
            )

        # ----------------------------------------------------
        # A2. Retirar o Top-4 da região boa
        # ----------------------------------------------------
        top4_reset = (
            good_background.copy()
        )

        top4_reset[
            top4
        ] = random_start[
            top4
        ]

        method = (
            "near_top4_reset_before"
        )

        if local_run_key(
            method,
            restart,
        ) not in local_completed:
            run = evaluate_without_optimization(
                method=method,
                restart=restart,
                theta_full=top4_reset,
                seed_simulator=(
                    random_seed
                    + 110_000
                    + restart
                ),
                background_id=background_id,
            )

            save_local_run(
                run
            )

        # ----------------------------------------------------
        # A3. Otimizar somente o Top-4 sobre o fundo bom
        # ----------------------------------------------------
        method = (
            "near_top4_reoptimized"
        )

        if local_run_key(
            method,
            restart,
        ) not in local_completed:
            run = run_active_cobyla(
                method=method,
                active_indices=top4,
                restart=restart,
                theta_initial_full=top4_reset,
                maxiter=local_active_budget,
            )

            run[
                "background_id"
            ] = background_id

            save_local_run(
                run
            )

        # ----------------------------------------------------
        # A4. Controle: retirar o Bottom-4
        # ----------------------------------------------------
        bottom4_reset = (
            good_background.copy()
        )

        bottom4_reset[
            bottom4
        ] = random_start[
            bottom4
        ]

        method = (
            "near_bottom4_reset_before"
        )

        if local_run_key(
            method,
            restart,
        ) not in local_completed:
            run = evaluate_without_optimization(
                method=method,
                restart=restart,
                theta_full=bottom4_reset,
                seed_simulator=(
                    random_seed
                    + 120_000
                    + restart
                ),
                background_id=background_id,
            )

            save_local_run(
                run
            )

        # ----------------------------------------------------
        # A5. Reotimizar somente o Bottom-4
        # ----------------------------------------------------
        method = (
            "near_bottom4_reoptimized"
        )

        if local_run_key(
            method,
            restart,
        ) not in local_completed:
            run = run_active_cobyla(
                method=method,
                active_indices=bottom4,
                restart=restart,
                theta_initial_full=bottom4_reset,
                maxiter=local_active_budget,
            )

            run[
                "background_id"
            ] = background_id

            save_local_run(
                run
            )

        print(
            "teste local concluído:",
            restart + 1,
            "/",
            followup_n_restarts,
        )


In [ ]:

# ============================================================
# 27. RESUMO PAREADO DO TESTE LOCAL
# ============================================================

local_results_df = pd.DataFrame(
    [
        {
            "method": run.get(
                "method"
            ),
            "restart": run.get(
                "restart"
            ),
            "background_id": run.get(
                "background_id"
            ),
            "status": run.get(
                "status"
            ),
            "energy": run.get(
                "objective_function_value"
            ),
            "energy_gap": run.get(
                "energy_gap"
            ),
            "nfev": run.get(
                "nfev_callback"
            ),
            "p_exact": run.get(
                "probability_best_answer"
            ),
            "dominant_bitstring": run.get(
                "most_frequent_bitstring"
            ),
            "is_exact_dominant": run.get(
                "is_exact_dominant"
            ),
            "is_near_optimal": run.get(
                "is_near_optimal"
            ),
        }
        for run in local_runs
    ]
)

local_results_df.to_csv(
    followup_output_dir
    / "near_background_local_results.csv",
    index=False,
)

local_summary_df = (
    local_results_df[
        local_results_df[
            "status"
        ]
        == "ok"
    ]
    .groupby(
        "method",
        as_index=False,
    )
    .agg(
        runs=(
            "restart",
            "count",
        ),
        gap_median=(
            "energy_gap",
            "median",
        ),
        gap_mean=(
            "energy_gap",
            "mean",
        ),
        p_exact_median=(
            "p_exact",
            "median",
        ),
        exact_rate=(
            "is_exact_dominant",
            "mean",
        ),
        near_rate=(
            "is_near_optimal",
            "mean",
        ),
        nfev_median=(
            "nfev",
            "median",
        ),
    )
)

local_summary_df[
    "exact_rate_pct"
] = (
    100
    * local_summary_df[
        "exact_rate"
    ]
)

local_summary_df.to_csv(
    followup_output_dir
    / "near_background_local_summary.csv",
    index=False,
)

display(
    local_summary_df.sort_values(
        "gap_median"
    )
)


def paired_local_comparison(
    before_method,
    after_method,
    label,
):
    before = (
        local_results_df[
            local_results_df[
                "method"
            ]
            == before_method
        ][
            [
                "restart",
                "energy_gap",
                "p_exact",
                "is_exact_dominant",
            ]
        ]
        .rename(
            columns={
                "energy_gap": (
                    "gap_before"
                ),
                "p_exact": (
                    "p_exact_before"
                ),
                "is_exact_dominant": (
                    "exact_before"
                ),
            }
        )
    )

    after = (
        local_results_df[
            local_results_df[
                "method"
            ]
            == after_method
        ][
            [
                "restart",
                "energy_gap",
                "p_exact",
                "is_exact_dominant",
                "nfev",
            ]
        ]
        .rename(
            columns={
                "energy_gap": (
                    "gap_after"
                ),
                "p_exact": (
                    "p_exact_after"
                ),
                "is_exact_dominant": (
                    "exact_after"
                ),
            }
        )
    )

    paired = before.merge(
        after,
        on="restart",
        validate="one_to_one",
    )

    paired[
        "delta_gap_after_minus_before"
    ] = (
        paired[
            "gap_after"
        ]
        - paired[
            "gap_before"
        ]
    )

    paired[
        "delta_p_exact_after_minus_before"
    ] = (
        paired[
            "p_exact_after"
        ]
        - paired[
            "p_exact_before"
        ]
    )

    paired[
        "group"
    ] = label

    return paired


top4_local_paired_df = paired_local_comparison(
    before_method="near_top4_reset_before",
    after_method="near_top4_reoptimized",
    label="top4",
)

bottom4_local_paired_df = paired_local_comparison(
    before_method="near_bottom4_reset_before",
    after_method="near_bottom4_reoptimized",
    label="bottom4",
)

local_paired_df = pd.concat(
    [
        top4_local_paired_df,
        bottom4_local_paired_df,
    ],
    ignore_index=True,
)

local_paired_df.to_csv(
    followup_output_dir
    / "near_background_paired_recovery.csv",
    index=False,
)

print(
    "Mediana da mudança do gap após reotimização:"
)

display(
    local_paired_df.groupby(
        "group",
        as_index=False,
    ).agg(
        median_delta_gap=(
            "delta_gap_after_minus_before",
            "median",
        ),
        median_delta_p_exact=(
            "delta_p_exact_after_minus_before",
            "median",
        ),
        exact_rate_after=(
            "exact_after",
            "mean",
        ),
        nfev_median=(
            "nfev",
            "median",
        ),
    )
)


In [ ]:

# ============================================================
# 28. EXPERIMENTO B — LIBERAÇÃO PROGRESSIVA
# ============================================================

# Ordem aninhada:
# 1 parâmetro  -> theta17
# 2 parâmetros -> theta17 + theta14
# 4 parâmetros -> theta17 + theta14 + theta02 + theta03
#
# Depois são adicionados os parâmetros restantes seguindo
# o ranking empírico.
progressive_original_order = [
    17,
    14,
    2,
    3,
] + [
    index
    for index
    in ranked_indices
    if index not in {
        17,
        14,
        2,
        3,
    }
]


def translate_ordered_indices(
    original_order,
):
    """
    Traduz o ranking preservando a ordem.
    """
    if (
        "theta_translation_df"
        not in globals()
        or theta_translation_df is None
    ):
        return [
            int(
                index
            )
            for index
            in original_order
        ]

    translation = dict(
        zip(
            theta_translation_df[
                "theta_index_original"
            ].astype(int),
            theta_translation_df[
                "theta_index_experiment"
            ].astype(int),
        )
    )

    return [
        int(
            translation[
                int(
                    index
                )
            ]
        )
        for index
        in original_order
    ]


progressive_order = translate_ordered_indices(
    progressive_original_order
)

progressive_dimensions = [
    1,
    2,
    4,
    6,
    8,
    12,
    16,
    30,
]

# As frações somam 1. O orçamento total permanece 250.
progressive_budget_fractions = np.asarray(
    [
        0.08,
        0.08,
        0.10,
        0.10,
        0.12,
        0.14,
        0.16,
        0.22,
    ],
    dtype=float,
)

progressive_stage_budgets = np.floor(
    progressive_budget_fractions
    * followup_total_budget
).astype(int)

progressive_stage_budgets[
    -1
] += (
    followup_total_budget
    - int(
        progressive_stage_budgets.sum()
    )
)

progressive_plan_df = pd.DataFrame(
    {
        "stage": np.arange(
            len(
                progressive_dimensions
            )
        ),
        "dimension": progressive_dimensions,
        "budget": progressive_stage_budgets,
        "active_indices": [
            progressive_order[
                :dimension
            ]
            for dimension
            in progressive_dimensions
        ],
    }
)

display(
    progressive_plan_df
)


In [ ]:

# ============================================================
# 29. OTIMIZAÇÃO DE UM ESTÁGIO SEM MEDIÇÃO INTERMEDIÁRIA
# ============================================================

def run_active_stage_without_sampling(
    active_indices,
    theta_initial_full,
    maxiter,
):
    """
    Executa um estágio do COBYLA e devolve o vetor completo.

    A medição por shots é feita somente no final da sequência
    progressiva. Assim, o método não paga oito medições extras.
    """
    global callback_list

    active_indices = sorted(
        set(
            int(
                index
            )
            for index
            in active_indices
        )
    )

    theta_initial_full = np.asarray(
        theta_initial_full,
        dtype=float,
    ).reshape(-1)

    fixed_reference = (
        theta_initial_full.copy()
    )

    x0_active = (
        theta_initial_full[
            active_indices
        ].copy()
    )

    callback_list = []

    def stage_objective(
        theta_active,
    ):
        theta_full = reconstruct_full_theta(
            theta_active=theta_active,
            active_indices=active_indices,
            fixed_reference=fixed_reference,
        )

        return exp_val(
            theta_full
        )

    optimizer = COBYLA(
        maxiter=int(
            maxiter
        )
    )

    start = perf_counter()

    result = optimizer.minimize(
        fun=stage_objective,
        x0=x0_active,
    )

    elapsed = (
        perf_counter()
        - start
    )

    theta_final_full = reconstruct_full_theta(
        theta_active=result.x,
        active_indices=active_indices,
        fixed_reference=fixed_reference,
    )

    energy = float(
        result.fun
    )

    return {
        "theta_final_full": theta_final_full,
        "energy": energy,
        "energy_gap": abs(
            energy
            - EXACT_ENERGY
        ),
        "nfev": len(
            callback_list
        ),
        "elapsed_s": elapsed,
        "callback_list": [
            (
                np.asarray(
                    theta,
                    dtype=float,
                ),
                float(
                    callback_energy
                ),
            )
            for theta, callback_energy
            in callback_list
        ],
    }


def run_progressive_release(
    restart,
    theta_initial_full,
):
    """
    Amplia progressivamente o conjunto ativo.

    O método para antecipadamente quando alcança o gap quase ótimo.
    O número total de avaliações nunca ultrapassa o orçamento
    followup_total_budget.
    """
    theta_current = np.asarray(
        theta_initial_full,
        dtype=float,
    ).copy()

    total_nfev = 0
    total_time = 0.0
    total_callback = []
    stage_rows = []

    for (
        stage_number,
        dimension,
        stage_budget,
    ) in zip(
        range(
            len(
                progressive_dimensions
            )
        ),
        progressive_dimensions,
        progressive_stage_budgets,
    ):
        remaining_budget = (
            followup_total_budget
            - total_nfev
        )

        if remaining_budget <= 0:
            break

        allowed_budget = int(
            min(
                int(
                    stage_budget
                ),
                int(
                    remaining_budget
                ),
            )
        )

        if allowed_budget <= 0:
            continue

        active_indices = (
            progressive_order[
                :dimension
            ]
        )

        stage_result = (
            run_active_stage_without_sampling(
                active_indices=active_indices,
                theta_initial_full=theta_current,
                maxiter=allowed_budget,
            )
        )

        theta_current = (
            stage_result[
                "theta_final_full"
            ].copy()
        )

        total_nfev += int(
            stage_result[
                "nfev"
            ]
        )

        total_time += float(
            stage_result[
                "elapsed_s"
            ]
        )

        total_callback.extend(
            stage_result[
                "callback_list"
            ]
        )

        stage_rows.append(
            {
                "restart": int(
                    restart
                ),
                "stage": int(
                    stage_number
                ),
                "dimension": int(
                    dimension
                ),
                "budget_requested": int(
                    allowed_budget
                ),
                "nfev_stage": int(
                    stage_result[
                        "nfev"
                    ]
                ),
                "nfev_cumulative": int(
                    total_nfev
                ),
                "energy": float(
                    stage_result[
                        "energy"
                    ]
                ),
                "energy_gap": float(
                    stage_result[
                        "energy_gap"
                    ]
                ),
                "active_indices": json.dumps(
                    active_indices
                ),
            }
        )

        if (
            stage_result[
                "energy_gap"
            ]
            < near_optimal_threshold
        ):
            break

    final_energy = float(
        exp_val(
            theta_current
        )
    )

    # A avaliação final acima serve somente para auditoria.
    # Ela não é adicionada ao orçamento do COBYLA.
    final_sampling = sample_final_solution(
        theta_full=theta_current,
        seed_simulator=(
            random_seed
            + 200_000
            + restart
        ),
    )

    return {
        "method": (
            "progressive_release"
        ),
        "restart": int(
            restart
        ),
        "status": "ok",
        "n_active": int(
            stage_rows[
                -1
            ][
                "dimension"
            ]
        ),
        "active_indices": json.loads(
            stage_rows[
                -1
            ][
                "active_indices"
            ]
        ),
        "theta_initial_full": np.asarray(
            theta_initial_full,
            dtype=float,
        ).copy(),
        "theta_final_full": theta_current,
        "objective_function_value": final_energy,
        "energy_gap": abs(
            final_energy
            - EXACT_ENERGY
        ),
        "is_near_optimal": (
            abs(
                final_energy
                - EXACT_ENERGY
            )
            < near_optimal_threshold
        ),
        "nfev_callback": int(
            total_nfev
        ),
        "nfev_result": int(
            total_nfev
        ),
        "time_spent_optimizer": float(
            total_time
        ),
        "callback_list": total_callback,
        "stage_rows": stage_rows,
        **final_sampling,
    }


In [ ]:

# ============================================================
# 30. COMPARAÇÃO PAREADA: FULL-30 X PROGRESSIVO
# ============================================================

progressive_checkpoint_path = (
    followup_output_dir
    / "progressive_paired_runs.pkl"
)

if (
    run_progressive_release_test
    and progressive_checkpoint_path.exists()
):
    progressive_runs = pd.read_pickle(
        progressive_checkpoint_path
    )

    if not isinstance(
        progressive_runs,
        list,
    ):
        progressive_runs = []
else:
    progressive_runs = []


progressive_completed = {
    (
        run.get(
            "method"
        ),
        int(
            run.get(
                "restart"
            )
        ),
    )
    for run in progressive_runs
    if run.get(
        "status"
    ) == "ok"
}


def save_progressive_run(
    run,
):
    key = (
        run[
            "method"
        ],
        int(
            run[
                "restart"
            ]
        ),
    )

    if key not in progressive_completed:
        progressive_runs.append(
            run
        )

        progressive_completed.add(
            key
        )

        pd.to_pickle(
            progressive_runs,
            progressive_checkpoint_path,
        )


if run_progressive_release_test:

    for restart in range(
        followup_n_restarts
    ):
        theta_start = (
            followup_initial_points[
                restart
            ].copy()
        )

        full_key = (
            "followup_full_30",
            restart,
        )

        if full_key not in progressive_completed:
            full_run = run_active_cobyla(
                method="followup_full_30",
                active_indices=list(
                    range(
                        n_parameters
                    )
                ),
                restart=restart,
                theta_initial_full=theta_start,
                maxiter=followup_total_budget,
            )

            save_progressive_run(
                full_run
            )

        progressive_key = (
            "progressive_release",
            restart,
        )

        if (
            progressive_key
            not in progressive_completed
        ):
            progressive_run = (
                run_progressive_release(
                    restart=restart,
                    theta_initial_full=theta_start,
                )
            )

            save_progressive_run(
                progressive_run
            )

        print(
            "comparação pareada concluída:",
            restart + 1,
            "/",
            followup_n_restarts,
        )


In [ ]:

# ============================================================
# 31. RESULTADOS DO MÉTODO PROGRESSIVO
# ============================================================

progressive_results_df = pd.DataFrame(
    [
        {
            "method": run.get(
                "method"
            ),
            "restart": run.get(
                "restart"
            ),
            "status": run.get(
                "status"
            ),
            "n_active_final": run.get(
                "n_active"
            ),
            "energy": run.get(
                "objective_function_value"
            ),
            "energy_gap": run.get(
                "energy_gap"
            ),
            "nfev": run.get(
                "nfev_callback"
            ),
            "optimizer_time": run.get(
                "time_spent_optimizer"
            ),
            "p_exact": run.get(
                "probability_best_answer"
            ),
            "is_exact_dominant": run.get(
                "is_exact_dominant"
            ),
            "is_near_optimal": run.get(
                "is_near_optimal"
            ),
            "dominant_bitstring": run.get(
                "most_frequent_bitstring"
            ),
        }
        for run in progressive_runs
    ]
)

progressive_results_df.to_csv(
    followup_output_dir
    / "progressive_paired_results.csv",
    index=False,
)

progressive_summary_df = (
    progressive_results_df[
        progressive_results_df[
            "status"
        ]
        == "ok"
    ]
    .groupby(
        "method",
        as_index=False,
    )
    .agg(
        runs=(
            "restart",
            "count",
        ),
        nfev_median=(
            "nfev",
            "median",
        ),
        nfev_mean=(
            "nfev",
            "mean",
        ),
        gap_median=(
            "energy_gap",
            "median",
        ),
        gap_mean=(
            "energy_gap",
            "mean",
        ),
        p_exact_median=(
            "p_exact",
            "median",
        ),
        exact_rate=(
            "is_exact_dominant",
            "mean",
        ),
        near_rate=(
            "is_near_optimal",
            "mean",
        ),
        final_dimension_median=(
            "n_active_final",
            "median",
        ),
    )
)

progressive_summary_df[
    "exact_rate_pct"
] = (
    100
    * progressive_summary_df[
        "exact_rate"
    ]
)

progressive_summary_df.to_csv(
    followup_output_dir
    / "progressive_paired_summary.csv",
    index=False,
)

display(
    progressive_summary_df
)


full_paired = (
    progressive_results_df[
        progressive_results_df[
            "method"
        ]
        == "followup_full_30"
    ][
        [
            "restart",
            "energy_gap",
            "nfev",
            "p_exact",
            "is_exact_dominant",
        ]
    ]
    .rename(
        columns={
            "energy_gap": (
                "gap_full"
            ),
            "nfev": (
                "nfev_full"
            ),
            "p_exact": (
                "p_exact_full"
            ),
            "is_exact_dominant": (
                "exact_full"
            ),
        }
    )
)

progressive_paired = (
    progressive_results_df[
        progressive_results_df[
            "method"
        ]
        == "progressive_release"
    ][
        [
            "restart",
            "energy_gap",
            "nfev",
            "p_exact",
            "is_exact_dominant",
            "n_active_final",
        ]
    ]
    .rename(
        columns={
            "energy_gap": (
                "gap_progressive"
            ),
            "nfev": (
                "nfev_progressive"
            ),
            "p_exact": (
                "p_exact_progressive"
            ),
            "is_exact_dominant": (
                "exact_progressive"
            ),
        }
    )
)

progressive_pairwise_df = (
    full_paired.merge(
        progressive_paired,
        on="restart",
        validate="one_to_one",
    )
)

progressive_pairwise_df[
    "nfev_reduction_pct"
] = (
    100
    * (
        1
        - progressive_pairwise_df[
            "nfev_progressive"
        ]
        / progressive_pairwise_df[
            "nfev_full"
        ].clip(
            lower=1
        )
    )
)

progressive_pairwise_df[
    "gap_ratio_progressive_over_full"
] = (
    progressive_pairwise_df[
        "gap_progressive"
    ]
    / progressive_pairwise_df[
        "gap_full"
    ].clip(
        lower=1e-15
    )
)

progressive_pairwise_df.to_csv(
    followup_output_dir
    / "progressive_pairwise_comparison.csv",
    index=False,
)

display(
    progressive_pairwise_df
)

print(
    "Redução mediana de nfev:",
    f"{progressive_pairwise_df['nfev_reduction_pct'].median():.2f}%",
)

print(
    "Razão mediana de gap progressivo/full:",
    f"{progressive_pairwise_df['gap_ratio_progressive_over_full'].median():.4f}",
)


In [ ]:

# ============================================================
# 32. TRAJETÓRIA DOS ESTÁGIOS E GRÁFICOS FINAIS
# ============================================================

stage_rows_all = []

for run in progressive_runs:
    if (
        run.get(
            "method"
        )
        != "progressive_release"
    ):
        continue

    stage_rows_all.extend(
        run.get(
            "stage_rows",
            [],
        )
    )

progressive_stages_df = pd.DataFrame(
    stage_rows_all
)

if not progressive_stages_df.empty:
    progressive_stages_df.to_csv(
        followup_output_dir
        / "progressive_stage_trajectories.csv",
        index=False,
    )

    stage_summary_df = (
        progressive_stages_df
        .groupby(
            "dimension",
            as_index=False,
        )
        .agg(
            gap_median=(
                "energy_gap",
                "median",
            ),
            gap_mean=(
                "energy_gap",
                "mean",
            ),
            nfev_cumulative_median=(
                "nfev_cumulative",
                "median",
            ),
        )
    )

    display(
        stage_summary_df
    )

    fig, ax = plt.subplots(
        figsize=(
            10,
            6,
        )
    )

    ax.plot(
        stage_summary_df[
            "dimension"
        ],
        stage_summary_df[
            "gap_median"
        ],
        marker="o",
        linewidth=2,
    )

    ax.axhline(
        near_optimal_threshold,
        linestyle="--",
        linewidth=1.5,
        label="limite quase ótimo",
    )

    ax.set_yscale(
        "log"
    )

    ax.set_xlabel(
        "Quantidade de theta liberados"
    )

    ax.set_ylabel(
        "Gap energético mediano"
    )

    ax.set_title(
        "Evolução da energia durante a liberação progressiva"
    )

    ax.grid(
        alpha=0.25
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        followup_output_dir
        / "01_progressive_gap_by_dimension.png",
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()


if (
    "local_summary_df"
    in globals()
    and not local_summary_df.empty
):
    plot_local = local_summary_df.sort_values(
        "gap_median"
    )

    fig, ax = plt.subplots(
        figsize=(
            12,
            6,
        )
    )

    ax.barh(
        plot_local[
            "method"
        ],
        plot_local[
            "gap_median"
        ],
    )

    ax.axvline(
        near_optimal_threshold,
        linestyle="--",
        linewidth=1.5,
    )

    ax.set_xscale(
        "log"
    )

    ax.set_xlabel(
        "Gap energético mediano"
    )

    ax.set_title(
        "Recuperação local sobre fundo quase ótimo"
    )

    ax.grid(
        axis="x",
        alpha=0.25,
    )

    fig.tight_layout()

    fig.savefig(
        followup_output_dir
        / "02_local_near_background_gap.png",
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()


if (
    "progressive_summary_df"
    in globals()
    and not progressive_summary_df.empty
):
    fig, ax = plt.subplots(
        figsize=(
            9,
            5,
        )
    )

    ax.bar(
        progressive_summary_df[
            "method"
        ],
        progressive_summary_df[
            "nfev_median"
        ],
    )

    ax.set_ylabel(
        "Mediana de avaliações da energia"
    )

    ax.set_title(
        "Orçamento usado: completo versus progressivo"
    )

    ax.tick_params(
        axis="x",
        rotation=15,
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    fig.tight_layout()

    fig.savefig(
        followup_output_dir
        / "03_full_vs_progressive_nfev.png",
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()



# Leitura dos novos resultados

## Teste do fundo quase ótimo

Compare, nesta ordem:

```text
near_good_background
near_top4_reset_before
near_top4_reoptimized
```

Um resultado favorável ao Top-4 ocorre quando:

\[
\Delta E_{\mathrm{reset}}
\gg
\Delta E_{\mathrm{reoptimized}},
\]

e a probabilidade do bitstring exato aumenta depois da reotimização.

O controle correspondente é:

```text
near_bottom4_reset_before
near_bottom4_reoptimized
```

O Top-4 deve recuperar mais energia ou mais probabilidade que o Bottom-4 para demonstrar vantagem específica.

## Teste progressivo

Compare:

```text
followup_full_30
progressive_release
```

Os dois métodos começam exatamente do mesmo vetor em cada `restart` e possuem o mesmo orçamento máximo de 250 avaliações.

O método progressivo é promissor quando:

1. usa menos avaliações efetivas;
2. mantém gap próximo ao `full_30`;
3. mantém a taxa do bitstring `1001011000`;
4. frequentemente para antes de liberar os 30 parâmetros.

## Execução robusta

Na célula 22, altere:

```python
followup_robust_mode = True
```

Isso executará 30 pares independentes. Os checkpoints são próprios desta etapa, portanto o processo pode ser retomado sem perder as execuções já concluídas.
